# 12. PITWALL Replay Engine & Strategy Replay Pipeline

This notebook runs the end-to-end replay engine for session 9558 (Silverstone 2024, 52 laps).
It demonstrates:
1. Loading validated session data and detecting clean laps, pit in/out laps, and finishers.
2. Fitting the lap fixed-effects pace model to isolate compound and degradation effects without weather confounding.
3. Measuring empirical pit loss across historical green-flag stops.
4. Generating procedural multi-stop strategy candidates (k=1 and k=2 stops).
5. Running the acceptance test suite (Tests A, B, C, D) with multi-stop baseline evaluation.
6. Re-running the original motivating scenarios: Driver 63 (Russell) vs Driver 44 (Hamilton) at lap 26.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pitwall.data import load_race, get_finishers, get_driver_laps
from pitwall.pace_model import PaceModel
from pitwall.pit_loss import measure_pit_loss
from pitwall.strategy_generator import generate_multi_stop_candidates
from pitwall.engine import replay, replay_actual, Stay, Pit, Strategy
from pitwall.validation import run_plumbing_test, run_identity_test, run_sanity_bound_test, run_support_test

# 1. Load Session 9558 Data
df = load_race(9558)
finishers = get_finishers(df)
print(f'Loaded session 9558: {len(df)} lap records across {df["driver_number"].nunique()} drivers.')
print(f'Finishers ({len(finishers)}): {finishers}')
print(f'Clean laps identified: {df["is_clean_lap"].sum()}')

Loaded session 9558: 962 lap records across 20 drivers.
Finishers (12): [1, 2, 4, 14, 18, 20, 22, 23, 27, 44, 55, 81]
Clean laps identified: 839


## 2. Fit Lap Fixed-Effects Pace Model

The model formula is:
$$\text{lap\_duration} \sim C(\text{lap\_number}) + C(\text{driver\_number}) + C(\text{compound}) + C(\text{compound}):\text{tyre\_age}$$
The per-lap term absorbs track wetness and fuel burn, preventing compound/tyre-age confounding.

In [2]:
production_pace_model = PaceModel(reference_compound='MEDIUM').fit(df)
print(f'Model R-squared: {production_pace_model.rsquared:.4f}')
rep = production_pace_model.identification_report
print(f'Clean laps: {rep.n_clean_laps}, Multi-compound laps: {rep.n_multi_compound_laps}')
print('\nCompound Pair Same-Lap Identification:')
for pair, cnt in rep.pair_counts.items():
    status = '[WEAK]' if pair in rep.weakly_identified_pairs else '[OK]'
    print(f'  {status} {pair}: {cnt} laps')

# Coefficient table
coef_data = []
for c in production_pace_model.coefficients:
    if 'compound' in c.param_name.lower():
        coef_data.append({'Parameter': c.param_name, 'Estimate': round(c.coef, 4), '95% CI Lower': round(c.ci_lower, 4), '95% CI Upper': round(c.ci_upper, 4), 'p-value': round(c.pvalue, 4)})
pd.DataFrame(coef_data)

Model R-squared: 0.9664
Clean laps: 839, Multi-compound laps: 37

Compound Pair Same-Lap Identification:
  [OK] MEDIUM vs HARD: 29 laps
  [WEAK] MEDIUM vs INTERMEDIATE: 7 laps
  [OK] MEDIUM vs SOFT: 30 laps
  [WEAK] HARD vs INTERMEDIATE: 0 laps
  [OK] HARD vs SOFT: 29 laps
  [WEAK] INTERMEDIATE vs SOFT: 0 laps


,Parameter,Estimate,95% CI Lower,95% CI Upper,p-value
0,C(compound)[T.HARD],-0.2033,-0.5442,0.1376,0.2425
1,C(compound)[T.INTERMEDIATE],10.2501,7.4275,13.0727,0.0000
2,C(compound)[T.SOFT],-0.2157,-0.7360,0.3046,0.4165
3,C(compound)[MEDIUM]:tyre_age,-0.1600,-0.2511,-0.0690,0.0006
4,C(compound)[HARD]:tyre_age,-0.1236,-0.2172,-0.0300,0.0096
5,C(compound)[INTERMEDIATE]:tyre_age,-0.5832,-0.7992,-0.3673,0.0000
6,C(compound)[SOFT]:tyre_age,-0.0531,-0.1507,0.0445,0.2863


In [3]:
# PITWALL: inspect the strategy-engine interfaces

import inspect
import pitwall.engine as engine
import pitwall.strategy_generator as generator

try:
    import pitwall.optimizer as optimizer
    print("Optimizer module:", optimizer.__file__)
    print("Optimizer functions:",
          [name for name, obj in inspect.getmembers(optimizer, inspect.isfunction)
           if not name.startswith("_")])
except ImportError:
    print("Optimizer module not found. Check that optimizer.py was copied.")

print("\n--- Replay interface ---")
print(inspect.signature(engine.replay))
print(inspect.signature(engine.replay_actual))

print("\n--- Candidate generator interface ---")
print(inspect.signature(generator.generate_multi_stop_candidates))

print("\n--- Pace model interface ---")
print("\n--- Pace model methods ---")
print([name for name in dir(production_pace_model) if not name.startswith("_")])

Optimizer module: C:\Users\Nihan\.vscode\pitwall\pitwall\optimizer.py
Optimizer functions: ['dataclass', 'generate_multi_stop_candidates', 'optimize_strategy', 'replay']

--- Replay interface ---
(session_df: pandas.DataFrame, target_driver: int, decision_lap: int, strategy: pitwall.engine.Stay | pitwall.engine.Pit | pitwall.engine.Strategy | None = None, pace_model=None, pit_loss: float = 22.0, track_mode: str = 'replay', race_length: int | None = None) -> pitwall.engine.ReplayResult | Dict[str, pitwall.engine.ReplayResult]
(session_df: pandas.DataFrame) -> pandas.DataFrame

--- Candidate generator interface ---
(race_df: pandas.DataFrame, target_driver: int, decision_lap: int, race_length: int | None = None, pit_window: int = 5, pit_intervals: tuple = (1, 2, 3, 4, 5), max_stops: int = 2, supported_compounds: Set[str] | None = None, min_stint_length: int = 5) -> pandas.DataFrame

--- Pace model interface ---

--- Pace model methods ---
['base_driver', 'base_lap', 'coefficients', 'comp

In [4]:

# PITWALL: connect the strategy optimiser

import inspect
from pitwall.optimizer import optimize_strategy

print("Optimizer signature:")
print(inspect.signature(optimize_strategy))

print("\nPace prediction signature:")
print(inspect.signature(production_pace_model.predict_lap))

print("\nReplay result structure:")
print(inspect.signature(replay))


Optimizer signature:
(session_df: 'pd.DataFrame', target_driver: 'int', decision_lap: 'int', pace_model: 'Any' = None, pit_loss: 'float' = 22.0, track_mode: 'str' = 'replay', max_stops: 'int' = 2, max_candidates: 'Optional[int]' = 95, supported_compounds: 'Optional[set]' = None) -> 'OptimizationResult'

Pace prediction signature:
(lap_number: int, driver_number: int, compound: str, tyre_age: int, mode: str = 'replay', decision_lap: int | None = None) -> float

Replay result structure:
(session_df: pandas.DataFrame, target_driver: int, decision_lap: int, strategy: pitwall.engine.Stay | pitwall.engine.Pit | pitwall.engine.Strategy | None = None, pace_model=None, pit_loss: float = 22.0, track_mode: str = 'replay', race_length: int | None = None) -> pitwall.engine.ReplayResult | Dict[str, pitwall.engine.ReplayResult]


In [5]:

# Display only the best strategies

result = optimize_strategy(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    pace_model=model,
    pit_loss=22.0,
    track_mode="replay",
    max_stops=2,
    max_candidates=95,
)

display(
    result.ranking[
        [
            "strategy_id",
            "pit_sequence_str",
            "num_stops",
            "projected_final_position",
            "projected_cumulative_time_s",
        ]
    ].head(10)
)

print("\nRecommended strategy:")
print(result.ranking.iloc[0])


NameError: name 'model' is not defined

In [ ]:
import inspect

print("Strategy:", inspect.signature(Strategy))
print("Pit:", inspect.signature(Pit))

Strategy: (pit_sequence: Tuple[Tuple[int, str], ...] = <factory>, description: str = '') -> None
Pit: (lap: int, compound: str) -> None


In [ ]:

# PITWALL: compare baseline against the recommended strategy

from pitwall.engine import replay, Strategy

recommended = result.ranking.iloc[0]

baseline_result = replay(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    strategy=None,
    pace_model=model,
    pit_loss=22.0,
    track_mode="replay",
)

recommended_strategy = Strategy(
    pit_sequence=tuple(recommended["pit_sequence"]),
    description=str(recommended["pit_sequence_str"]),
)

recommended_result = replay(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    strategy=recommended_strategy,
    pace_model=model,
    pit_loss=22.0,
    track_mode="replay",
)


def compact_summary(label, output):
    print(f"\n{label}")
    
    if isinstance(output, dict):
        for driver, item in output.items():
            print(f"Driver: {driver}")
            print(item)
        return

    for field in [
        "projected_final_position",
        "projected_cumulative_time_s",
        "time_delta_vs_actual_s",
        "positions_gained_vs_actual",
        "ground_truth_available",
    ]:
        if hasattr(output, field):
            print(f"{field}: {getattr(output, field)}")

compact_summary("BASELINE", baseline_result)
compact_summary("RECOMMENDED", recommended_result)



AttributeError: 'ReplayResult' object has no attribute 'ranking'

In [ ]:

# PITWALL: concise baseline vs recommended comparison

from pitwall.engine import replay, Strategy

baseline_row = result.ranking.loc[
    result.ranking["is_baseline"]
].iloc[0]

best_row = result.ranking.iloc[0]

def replay_summary(row):
    strategy = Strategy(
        pit_sequence=tuple(row["pit_sequence"]),
        description=str(row["pit_sequence_str"]),
    )

    output = replay(
        session_df=df,
        target_driver=44,
        decision_lap=26,
        strategy=strategy,
        pace_model=model,
        pit_loss=22.0,
        track_mode="replay",
    )

    return output.summary

baseline = replay_summary(baseline_row)
best = replay_summary(best_row)

for label, summary in [
    ("BASELINE", baseline),
    ("RECOMMENDED", best),
]:
    print(f"\n{label}")
    for key in [
        "final_position",
        "final_cumulative_time",
        "total_time_delta_vs_actual",
        "positions_gained_vs_actual",
    ]:
        print(f"{key}: {summary.get(key)}")

print(
    "\nProjected time difference (seconds):",
    round(
        best["final_cumulative_time"]
        - baseline["final_cumulative_time"],
        3,
    ),
)



BASELINE
final_position: 1
final_cumulative_time: 4945.869118444601
total_time_delta_vs_actual: -1.5578815553999448
positions_gained_vs_actual: 0

RECOMMENDED
final_position: 1
final_cumulative_time: 4850.312686651329
total_time_delta_vs_actual: -97.1143133486712
positions_gained_vs_actual: 0

Projected time difference (seconds): -95.556


In [ ]:
import inspect
from pitwall.pace_model import PaceModel
from pitwall.engine import replay
from pitwall.optimizer import optimize_strategy

print("PACE MODEL")
print(inspect.signature(model.predict_lap))

print("\nREPLAY")
print(inspect.signature(replay))

print("\nOPTIMIZER")
print(inspect.signature(optimize_strategy))

print("\nENGINE SOURCE")
print(inspect.getsource(replay))

PACE MODEL
(lap_number: int, driver_number: int, compound: str, tyre_age: int, mode: str = 'replay', decision_lap: int | None = None) -> float

REPLAY
(session_df: pandas.DataFrame, target_driver: int, decision_lap: int, strategy: pitwall.engine.Stay | pitwall.engine.Pit | pitwall.engine.Strategy | None = None, pace_model=None, pit_loss: float = 22.0, track_mode: str = 'replay') -> pitwall.engine.ReplayResult | Dict[str, pitwall.engine.ReplayResult]

OPTIMIZER
(session_df: 'pd.DataFrame', target_driver: 'int', decision_lap: 'int', pace_model: 'Any' = None, pit_loss: 'float' = 22.0, track_mode: 'str' = 'replay', max_stops: 'int' = 2, max_candidates: 'Optional[int]' = 95, supported_compounds: 'Optional[set]' = None) -> 'OptimizationResult'

ENGINE SOURCE
def replay(
    session_df: pd.DataFrame,
    target_driver: int,
    decision_lap: int,
    strategy: Optional[Action] = None,
    pace_model=None,
    pit_loss: float = 22.0,
    track_mode: str = "replay",
) -> Union[ReplayResult, Dic

In [ ]:

from pitwall.engine import replay, Stay, Strategy

baseline_causal = replay(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    strategy=Stay(),
    pace_model=model,
    pit_loss=22.0,
    track_mode="causal",
)

best_causal = replay(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    strategy=Strategy(
        pit_sequence=tuple(best_row["pit_sequence"]),
        description=str(best_row["pit_sequence_str"]),
    ),
    pace_model=model,
    pit_loss=22.0,
    track_mode="causal",
)

print("BASELINE CAUSAL")
print(baseline_causal.summary)

print("\nRECOMMENDED CAUSAL")
print(best_causal.summary)

print("\nPROJECTED TIME DIFFERENCE")
print(round(
    best_causal.summary["final_cumulative_time"]
    - baseline_causal.summary["final_cumulative_time"],
    3
), "seconds")


BASELINE CAUSAL
{'target_driver': 44, 'decision_lap': 26, 'ground_truth_available': True, 'final_position': 4, 'final_cumulative_time': 4980.937245905431, 'total_time_delta_vs_actual': 33.510245905430565, 'positions_gained_vs_actual': -3}

RECOMMENDED CAUSAL
{'target_driver': 44, 'decision_lap': 26, 'ground_truth_available': True, 'final_position': 17, 'final_cumulative_time': 5151.090226306743, 'total_time_delta_vs_actual': 203.66322630674222, 'positions_gained_vs_actual': -16}

PROJECTED TIME DIFFERENCE
170.153 seconds


In [ ]:
from pitwall.engine import _project_rival_lap_time
print("BASELINE STRATEGY")
print(baseline_row["pit_sequence_str"])

print("\nRECOMMENDED STRATEGY")
print(best_row["pit_sequence_str"])

print("\nRIVAL FORECAST CHECK")

for driver in [1, 4, 16, 44, 63]:
    if driver == 44:
        continue

    print(f"\nDriver {driver}")

    for lap in [27, 30, 35, 40, 45, 50]:
        if lap > df["lap_number"].max():
            continue

        value = _project_rival_lap_time(
            df=df,
            driver_number=driver,
            decision_lap=26,
            lap_number=lap,
            pace_model=model,
        )

        print(f"Lap {lap}: {value:.3f}s")

BASELINE STRATEGY
INTERMEDIATE lap 27 -> SOFT lap 38

RECOMMENDED STRATEGY
INTERMEDIATE lap 37 -> SOFT lap 40

RIVAL FORECAST CHECK

Driver 1
Lap 27: 100.667s
Lap 30: 100.187s
Lap 35: 99.386s
Lap 40: 98.586s
Lap 45: 97.786s
Lap 50: 96.986s

Driver 4
Lap 27: 100.282s
Lap 30: 99.802s
Lap 35: 99.002s
Lap 40: 98.201s
Lap 45: 97.401s
Lap 50: 96.601s

Driver 16
Lap 27: 111.365s
Lap 30: 109.616s
Lap 35: 106.700s
Lap 40: 103.783s
Lap 45: 100.867s
Lap 50: 97.951s

Driver 63
Lap 27: 100.554s
Lap 30: 100.074s
Lap 35: 99.274s
Lap 40: 98.474s
Lap 45: 97.673s
Lap 50: 96.873s


In [ ]:
import inspect
import pitwall.engine as engine

print("ENGINE:", engine.__file__)

print("\nFUNCTIONS:")
print("replay:", hasattr(engine, "replay"))
print("helper:", hasattr(engine, "_project_rival_lap_time"))
print("replay_actual:", hasattr(engine, "replay_actual"))

print("\nMODES:")
print("replay:", model.predict_lap(
    lap_number=27,
    driver_number=44,
    compound="MEDIUM",
    tyre_age=1,
    mode="replay",
))

print("causal:", model.predict_lap(
    lap_number=27,
    driver_number=44,
    compound="MEDIUM",
    tyre_age=1,
    mode="causal",
    decision_lap=26,
))

ENGINE: C:\Users\Nihan\.vscode\pitwall\pitwall\engine.py

FUNCTIONS:
replay: True
helper: True
replay_actual: True

MODES:
replay: 109.5225539324548
causal: 104.27660925832807


In [ ]:
from pitwall.engine import replay, Stay

test_result = replay(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    strategy=Stay(),
    pace_model=model,
    pit_loss=22.0,
    track_mode="causal",
)

print(test_result.summary)
print("\nResult table shape:", test_result.laps_table.shape)

{'target_driver': 44, 'decision_lap': 26, 'ground_truth_available': True, 'final_position': 4, 'final_cumulative_time': 4980.937245905431, 'total_time_delta_vs_actual': 33.510245905430565, 'positions_gained_vs_actual': -3}

Result table shape: (1013, 12)


In [ ]:

from pitwall.rival_state import build_rival_state

rival_state = build_rival_state(
    session_df=df,
    target_driver=44,
    decision_lap=26,
)

display(rival_state)


print("Rivals captured:", len(rival_state))

print(
    "\nMaximum observed lap:",
    rival_state["lap_observed"].max()
)

print(
    "\nPit stops observed:"
)
display(
    rival_state[
        ["driver_number", "compound", "tyre_age",
         "pit_stops_observed", "stint_length",
         "last_pit_lap", "position"]
    ]
)

assert (rival_state["lap_observed"] <= 26).all()

assert 44 not in rival_state["driver_number"].values

print("\nRIVAL STATE CHECKS PASSED")



,driver_number,lap_observed,laps_behind_target,position,compound,tyre_age,stint_number,pit_stops_observed,stint_start_lap,stint_length,last_pit_lap,gap_to_target_s
0,4,26,0,1.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,-2.861
1,81,26,0,2.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,-2.279
2,63,26,0,4.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,1.342
3,1,26,0,5.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,9.478
4,55,26,0,6.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,10.661
5,27,26,0,7.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,27.549
6,14,26,0,8.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,31.378
7,18,26,0,9.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,29.637
8,22,26,0,10.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,32.840
9,23,26,0,11.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,33.829


Rivals captured: 19

Maximum observed lap: 26

Pit stops observed:


,driver_number,compound,tyre_age,pit_stops_observed,stint_length,last_pit_lap,position
0,4,MEDIUM,25.0,0.0,26.0,NaN,1.0
1,81,MEDIUM,25.0,0.0,26.0,NaN,2.0
2,63,MEDIUM,25.0,0.0,26.0,NaN,4.0
3,1,MEDIUM,25.0,0.0,26.0,NaN,5.0
4,55,MEDIUM,25.0,0.0,26.0,NaN,6.0
5,27,MEDIUM,25.0,0.0,26.0,NaN,7.0
6,14,MEDIUM,25.0,0.0,26.0,NaN,8.0
7,18,MEDIUM,25.0,0.0,26.0,NaN,9.0
8,22,MEDIUM,25.0,0.0,26.0,NaN,10.0
9,23,MEDIUM,25.0,0.0,26.0,NaN,11.0



RIVAL STATE CHECKS PASSED


In [ ]:

for driver in [10, 16, 24, 31]:
    print(f"\nDRIVER {driver}")

    display(
        df.loc[
            (df["driver_number"] == driver) &
            (df["lap_number"] <= 26),
            [
                "driver_number",
                "lap_number",
                "stint_number",
                "compound",
                "tyre_age",
                "position",
            ]
        ].sort_values("lap_number")
    )



DRIVER 10


,driver_number,lap_number,stint_number,compound,tyre_age,position
207,10,1,NaN,NaN,NaN,19.0



DRIVER 16


,driver_number,lap_number,stint_number,compound,tyre_age,position
310,16,1,1.0,MEDIUM,0.0,11.0
311,16,2,1.0,MEDIUM,1.0,8.0
312,16,3,1.0,MEDIUM,2.0,8.0
313,16,4,1.0,MEDIUM,3.0,8.0
314,16,5,1.0,MEDIUM,4.0,8.0
315,16,6,1.0,MEDIUM,5.0,8.0
316,16,7,1.0,MEDIUM,6.0,8.0
317,16,8,1.0,MEDIUM,7.0,8.0
318,16,9,1.0,MEDIUM,8.0,8.0
319,16,10,1.0,MEDIUM,9.0,8.0



DRIVER 24


,driver_number,lap_number,stint_number,compound,tyre_age,position
569,24,1,1.0,SOFT,0.0,14.0
570,24,2,1.0,SOFT,1.0,14.0
571,24,3,1.0,SOFT,2.0,14.0
572,24,4,1.0,SOFT,3.0,14.0
573,24,5,1.0,SOFT,4.0,14.0
574,24,6,1.0,SOFT,5.0,14.0
575,24,7,1.0,SOFT,6.0,14.0
576,24,8,1.0,SOFT,7.0,15.0
577,24,9,1.0,SOFT,8.0,18.0
578,24,10,1.0,SOFT,9.0,18.0



DRIVER 31


,driver_number,lap_number,stint_number,compound,tyre_age,position
671,31,1,1.0,SOFT,0.0,18.0
672,31,2,1.0,SOFT,1.0,16.0
673,31,3,1.0,SOFT,2.0,16.0
674,31,4,1.0,SOFT,3.0,16.0
675,31,5,1.0,SOFT,4.0,16.0
676,31,6,1.0,SOFT,5.0,16.0
677,31,7,1.0,SOFT,6.0,16.0
678,31,8,1.0,SOFT,7.0,18.0
679,31,9,1.0,SOFT,8.0,17.0
680,31,10,1.0,SOFT,9.0,17.0


In [ ]:

from pitwall.rival_strategy import generate_rival_strategies

rival_strategies = generate_rival_strategies(
    rival_state=rival_state,
    decision_lap=26,
    race_end_lap=52,
    supported_compounds=[
        "SOFT",
        "MEDIUM",
        "HARD",
        "INTERMEDIATE",
        "WET",
    ],
    max_stops=2,
    pit_window_step=3,
    min_stint_length=3,
    max_candidates_per_rival=100,
)

print("Total rival scenarios:", len(rival_strategies))
print(
    "Rivals represented:",
    rival_strategies["driver_number"].nunique()
)

display(rival_strategies.head(20))


driver_4_scenarios = rival_strategies[
    rival_strategies["driver_number"] == 4
].copy()

display(
    driver_4_scenarios[
        [
            "strategy_id",
            "current_compound",
            "current_tyre_age",
            "observed_stops",
            "additional_stops",
            "pit_laps",
            "compounds_after_stops",
        ]
    ]
)


Total rival scenarios: 1800
Rivals represented: 18


,strategy_id,driver_number,current_compound,current_tyre_age,observed_stops,additional_stops,total_projected_stops,pit_laps,compounds_after_stops,decision_lap,race_end_lap
0,D4_S001,4,MEDIUM,25.0,0,0,0,(),(),26,52
1,D4_S002,4,MEDIUM,25.0,0,1,1,"(26,)","(SOFT,)",26,52
2,D4_S003,4,MEDIUM,25.0,0,1,1,"(26,)","(HARD,)",26,52
3,D4_S004,4,MEDIUM,25.0,0,1,1,"(26,)","(INTERMEDIATE,)",26,52
4,D4_S005,4,MEDIUM,25.0,0,1,1,"(26,)","(WET,)",26,52
5,D4_S006,4,MEDIUM,25.0,0,1,1,"(29,)","(SOFT,)",26,52
6,D4_S007,4,MEDIUM,25.0,0,1,1,"(29,)","(HARD,)",26,52
7,D4_S008,4,MEDIUM,25.0,0,1,1,"(29,)","(INTERMEDIATE,)",26,52
8,D4_S009,4,MEDIUM,25.0,0,1,1,"(29,)","(WET,)",26,52
9,D4_S010,4,MEDIUM,25.0,0,1,1,"(32,)","(SOFT,)",26,52


,strategy_id,current_compound,current_tyre_age,observed_stops,additional_stops,pit_laps,compounds_after_stops
0,D4_S001,MEDIUM,25.0,0,0,(),()
1,D4_S002,MEDIUM,25.0,0,1,"(26,)","(SOFT,)"
2,D4_S003,MEDIUM,25.0,0,1,"(26,)","(HARD,)"
3,D4_S004,MEDIUM,25.0,0,1,"(26,)","(INTERMEDIATE,)"
4,D4_S005,MEDIUM,25.0,0,1,"(26,)","(WET,)"
...,...,...,...,...,...,...,...
95,D4_S096,MEDIUM,25.0,0,2,"(41, 47)","(HARD, SOFT)"
96,D4_S097,MEDIUM,25.0,0,2,"(41, 47)","(INTERMEDIATE, INTERMEDIATE)"
97,D4_S098,MEDIUM,25.0,0,2,"(44, 47)","(SOFT, HARD)"
98,D4_S099,MEDIUM,25.0,0,2,"(44, 47)","(INTERMEDIATE, SOFT)"


In [ ]:

from pitwall.rival_strategy import generate_rival_strategies

rival_strategies = generate_rival_strategies(
    rival_state=rival_state,
    decision_lap=26,
    race_end_lap=52,
    supported_compounds=[
        "SOFT",
        "MEDIUM",
        "HARD",
    ],
    max_stops=2,
    pit_window_step=3,
    min_stint_length=5,
    max_candidates_per_rival=100,
)

print("Total scenarios:", len(rival_strategies))
print("Rivals:", rival_strategies["driver_number"].nunique())

display(
    rival_strategies[
        rival_strategies["driver_number"] == 4
    ].head(20)
)


assert (
    rival_strategies["additional_stops"] <= 2
).all()

assert (
    rival_strategies["pit_laps"].apply(
        lambda x: all(
            26 <= lap < 52 for lap in x
        )
    ).all()
    
assert (
    rival_strategies["compounds_after_stops"].apply(
        lambda x: all(
            c in ["SOFT", "MEDIUM", "HARD"]
            for c in x
        )
    ).all()

assert (
    rival_strategies.groupby("driver_number").size().max()
    <= 100
)

print("STAGE 2A VALIDATION PASSED")


SyntaxError: invalid syntax (3195519188.py, line 39)

In [ ]:
from pitwall.rival_probability import (
    assign_rival_strategy_probabilities
)

rival_strategy_probs = assign_rival_strategy_probabilities(
    rival_strategies
)

print("Total strategies:", len(rival_strategy_probs))
print(
    "Rivals:",
    rival_strategy_probs["driver_number"].nunique()
)

display(
    rival_strategy_probs[
        [
            "driver_number",
            "strategy_id",
            "additional_stops",
            "pit_laps",
            "compounds_after_stops",
            "strategy_probability",
        ]
    ].head(15)
)

# Every rival's probabilities should sum to 1.
probability_check = (
    rival_strategy_probs
    .groupby("driver_number")["strategy_probability"]
    .sum()
)

display(probability_check)

assert np.allclose(probability_check.values, 1.0)

# Inspect the most probable strategies for each rival.
top_strategies = (
    rival_strategy_probs
    .sort_values(
        ["driver_number", "strategy_probability"],
        ascending=[True, False]
    )
    .groupby("driver_number")
    .head(3)
)

display(
    top_strategies[
        [
            "driver_number",
            "strategy_id",
            "additional_stops",
            "pit_laps",
            "compounds_after_stops",
            "strategy_probability",
        ]
    ]
)

Total strategies: 1800
Rivals: 18


,driver_number,strategy_id,additional_stops,pit_laps,compounds_after_stops,strategy_probability
0,4,D4_S001,0,(),(),0.150000
1,4,D4_S002,1,"(26,)","(SOFT,)",0.020313
2,4,D4_S003,1,"(26,)","(HARD,)",0.020313
3,4,D4_S004,1,"(26,)","(INTERMEDIATE,)",0.020313
4,4,D4_S005,1,"(26,)","(WET,)",0.020313
5,4,D4_S006,1,"(29,)","(SOFT,)",0.020313
6,4,D4_S007,1,"(29,)","(HARD,)",0.020313
7,4,D4_S008,1,"(29,)","(INTERMEDIATE,)",0.020313
8,4,D4_S009,1,"(29,)","(WET,)",0.020313
9,4,D4_S010,1,"(32,)","(SOFT,)",0.020313


driver_number
1     1.0
2     1.0
3     1.0
4     1.0
11    1.0
14    1.0
16    1.0
18    1.0
20    1.0
22    1.0
23    1.0
24    1.0
27    1.0
31    1.0
55    1.0
63    1.0
77    1.0
81    1.0
Name: strategy_probability, dtype: float64

,driver_number,strategy_id,additional_stops,pit_laps,compounds_after_stops,strategy_probability
300,1,D1_S001,0,(),(),0.150000
301,1,D1_S002,1,"(26,)","(SOFT,)",0.020313
302,1,D1_S003,1,"(26,)","(HARD,)",0.020313
1100,2,D2_S001,0,(),(),0.150000
1101,2,D2_S002,1,"(26,)","(SOFT,)",0.020313
1102,2,D2_S003,1,"(26,)","(HARD,)",0.020313
1200,3,D3_S001,0,(),(),0.150000
1201,3,D3_S002,1,"(26,)","(SOFT,)",0.020313
1202,3,D3_S003,1,"(26,)","(HARD,)",0.020313
0,4,D4_S001,0,(),(),0.150000


In [ ]:
from pitwall.rival_probability import assign_rival_strategy_probabilities

rival_strategy_probs = assign_rival_strategy_probabilities(
    rival_strategies
)

print("Strategies:", len(rival_strategy_probs))
print("Rivals:", rival_strategy_probs["driver_number"].nunique())

Strategies: 1800
Rivals: 18


In [ ]:
from pitwall.rival_scenario import simulate_rival_scenarios

scenario_results, rival_summary = simulate_rival_scenarios(
    rival_state=rival_state,
    rival_strategy_probs=rival_strategy_probs,
    pace_model=model,
    decision_lap=26,
    race_end_lap=52,
    pit_loss=22.0,
)

print("Simulated scenarios:", len(scenario_results))
print("Rivals simulated:", rival_summary["driver_number"].nunique())

display(rival_summary.head(20))

display(
    scenario_results[
        [
            "driver_number",
            "strategy_id",
            "additional_stops",
            "pit_laps",
            "strategy_probability",
            "projected_future_time_s",
            "projected_pit_loss_s",
        ]
    ].head(20)
)

# Every simulated strategy must have a valid probability.
assert scenario_results["strategy_probability"].between(0, 1).all()

# No negative projected times.
assert (scenario_results["projected_future_time_s"] > 0).all()

# Every rival should have at least one simulated scenario.
assert rival_summary["scenario_count"].min() >= 1

print("All rival scenario checks passed.")

display(rival_summary)

Simulated scenarios: 1800
Rivals simulated: 18


,driver_number,scenario_count,expected_future_time_s,best_case_time_s,worst_case_time_s,most_likely_strategy
0,1,100,2692.790795,2565.321050,2927.549820,D1_S001
1,2,100,2721.411774,2593.942029,2956.170799,D2_S001
2,3,100,2736.804543,2609.334797,2971.563568,D3_S001
3,4,100,2682.785232,2555.315487,2917.544258,D4_S001
4,11,100,2765.953302,2703.678210,2890.666448,D11_S001
5,14,100,2712.799047,2585.329302,2947.558072,D14_S001
6,16,100,2763.248629,2700.973538,2887.961775,D16_S001
7,18,100,2716.471957,2589.002212,2951.230983,D18_S001
8,20,100,2720.741856,2593.272111,2955.500882,D20_S001
9,22,100,2718.821382,2591.351637,2953.580408,D22_S001


,driver_number,strategy_id,additional_stops,pit_laps,strategy_probability,projected_future_time_s,projected_pit_loss_s
0,4,D4_S001,0,(),0.150000,2555.315487,0.0
1,4,D4_S002,1,"(26,)",0.020313,2713.276337,22.0
2,4,D4_S003,1,"(26,)",0.020313,2688.858438,22.0
3,4,D4_S004,1,"(26,)",0.020313,2799.308176,22.0
4,4,D4_S005,1,"(26,)",0.020313,2737.525378,22.0
5,4,D4_S006,1,"(29,)",0.020313,2704.942494,22.0
6,4,D4_S007,1,"(29,)",0.020313,2685.773787,22.0
7,4,D4_S008,1,"(29,)",0.020313,2799.337445,22.0
8,4,D4_S009,1,"(29,)",0.020313,2724.561341,22.0
9,4,D4_S010,1,"(32,)",0.020313,2694.690230,22.0


All rival scenario checks passed.


,driver_number,scenario_count,expected_future_time_s,best_case_time_s,worst_case_time_s,most_likely_strategy
0,1,100,2692.790795,2565.321050,2927.549820,D1_S001
1,2,100,2721.411774,2593.942029,2956.170799,D2_S001
2,3,100,2736.804543,2609.334797,2971.563568,D3_S001
3,4,100,2682.785232,2555.315487,2917.544258,D4_S001
4,11,100,2765.953302,2703.678210,2890.666448,D11_S001
5,14,100,2712.799047,2585.329302,2947.558072,D14_S001
6,16,100,2763.248629,2700.973538,2887.961775,D16_S001
7,18,100,2716.471957,2589.002212,2951.230983,D18_S001
8,20,100,2720.741856,2593.272111,2955.500882,D20_S001
9,22,100,2718.821382,2591.351637,2953.580408,D22_S001


In [ ]:
import importlib
import pitwall.rival_scenario as rs

importlib.invalidate_caches()
rs = importlib.reload(rs)

print("Module:", rs.__file__)
print("Function available:", hasattr(rs, "simulate_rival_scenarios"))
print([
    name for name in dir(rs)
    if callable(getattr(rs, name)) and not name.startswith("_")
])

simulate_rival_scenarios = rs.simulate_rival_scenarios

Module: C:\Users\Nihan\.vscode\pitwall\pitwall\rival_scenario.py
Function available: True
['simulate_rival_scenarios']


In [ ]:
from pitwall.engine import replay

own_replay_results = replay(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    strategy=None,
    pace_model=model,
    pit_loss=22.0,
    track_mode="causal",
)

print("Target strategies:", len(own_replay_results))



Target strategies: 90


In [ ]:
import importlib
import pitwall.strategy_evaluator as se

importlib.invalidate_caches()
se = importlib.reload(se)

print("Function available:",
      hasattr(se, "evaluate_strategies_against_rivals"))

strategy_evaluation = se.evaluate_strategies_against_rivals(
    own_replay_results=own_replay_results,
    rival_scenario_results=scenario_results,
    rival_state=rival_state,
    session_df=df,
    target_driver=44,
    decision_lap=26,
    n_scenarios=1000,
)

display(strategy_evaluation.head(15))

Function available: True


,strategy_rank,strategy_id,projected_finish_time_s,win_probability,podium_probability,top5_probability,expected_finish_position,median_finish_position,worst_decile_position,scenarios_evaluated,rivals_included
0,1,S038,5151.090226,0.0,0.010,0.158,7.345,7.0,10.0,1000,18
1,2,S036,5155.945919,0.0,0.006,0.127,7.617,8.0,10.0,1000,18
2,3,S032,5160.588454,0.0,0.005,0.094,7.894,8.0,10.0,1000,18
3,4,S037,5164.769147,0.0,0.003,0.065,8.201,8.0,11.0,1000,18
4,5,S030,5165.017831,0.0,0.002,0.064,8.209,8.0,11.0,1000,18
5,6,S089,5168.619039,0.0,0.000,0.059,8.457,8.0,11.0,1000,18
6,7,S034,5169.464790,0.0,0.000,0.054,8.523,9.0,11.0,1000,18
7,8,S090,5169.944389,0.0,0.000,0.052,8.567,9.0,11.0,1000,18
8,9,S028,5173.947275,0.0,0.000,0.040,8.819,9.0,11.0,1000,18
9,10,S035,5177.704770,0.0,0.000,0.023,9.190,9.0,11.0,1000,18


In [ ]:
# Target's actual elapsed time at the decision lap
target_rows = df[
    (df["driver_number"] == 44)
    & (df["lap_number"] <= 26)
]

target_elapsed = target_rows["lap_duration"].sum()

# Baseline Stay strategy
baseline = own_replay_results["Stay"]

print("Target elapsed at lap 26:", target_elapsed)
print("Baseline projected finish time:",
      baseline.summary["final_cumulative_time"])

print("\nTarget state:")
print(
    rival_state.loc[
        rival_state["driver_number"] == 44
    ]
)

print("\nRival gap range:")
print(rival_state["gap_to_target_s"].describe())

KeyError: 'Stay'

In [ ]:
print(
    rival_strategy_probs.columns.tolist()
)

print(
    rival_strategy_probs[
        (rival_strategy_probs["driver_number"] == 44)
    ].head()
)

for strategy_id, result in list(own_replay_results.items())[:5]:
    print(strategy_id, result.summary)

import pitwall.strategy_generator as sg
import inspect

print("Module:", sg.__file__)

print("\nAvailable functions:")
for name, obj in inspect.getmembers(sg, inspect.isfunction):
    if obj.__module__ == sg.__name__:
        print(name, inspect.signature(obj))

['strategy_id', 'driver_number', 'current_compound', 'current_tyre_age', 'observed_stops', 'additional_stops', 'total_projected_stops', 'pit_laps', 'compounds_after_stops', 'decision_lap', 'race_end_lap', 'stop_count_probability', 'timing_probability_given_stops', 'compound_probability_given_timing', 'strategy_probability', 'probability_method']
Empty DataFrame
Columns: [strategy_id, driver_number, current_compound, current_tyre_age, observed_stops, additional_stops, total_projected_stops, pit_laps, compounds_after_stops, decision_lap, race_end_lap, stop_count_probability, timing_probability_given_stops, compound_probability_given_timing, strategy_probability, probability_method]
Index: []
S001 {'target_driver': 44, 'decision_lap': 26, 'ground_truth_available': True, 'final_position': 17, 'final_cumulative_time': 5246.646658100015, 'total_time_delta_vs_actual': 299.2196581000144, 'positions_gained_vs_actual': -16}
S002 {'target_driver': 44, 'decision_lap': 26, 'ground_truth_available':

In [ ]:
import inspect
import pitwall.strategy_generator as sg

print(inspect.getsource(sg.generate_multi_stop_candidates))

def generate_multi_stop_candidates(
    race_df: pd.DataFrame,
    target_driver: int,
    decision_lap: int,
    race_length: int = 52,
    pit_window: int = 5,
    pit_intervals: tuple = (1, 2, 3, 4, 5),
    max_stops: int = 2,
    supported_compounds: Optional[Set[str]] = None,
) -> pd.DataFrame:
    """Generate multi-stop strategy candidates as a table.
    
    Output table columns:
    - strategy_id: "S001", "S002", ...
    - pit_sequence: tuple of (lap, compound) tuples, e.g. ((27, 'INTERMEDIATE'), (38, 'SOFT'))
    - pit_sequence_str: human-readable, e.g. "INTER lap 27 -> SOFT lap 38"
    - is_baseline: True if this is the real historical sequence for this driver
    - baseline_distance: Manhattan/lag distance from real sequence (0 for baseline)
    - current_compound: driver's compound at decision_lap
    - num_stops: 1 or 2
    - description: human-readable narrative
    
    Design assumptions & heuristics:
    - Weather heuristic for two-stop: If currently wet at decision l

In [ ]:
from pitwall.strategy_generator import generate_multi_stop_candidates

target_candidates = generate_multi_stop_candidates(
    candidates = generate_multi_stop_candidates(
    race_df=df,
    target_driver=target_driver,
    decision_lap=decision_lap,
    race_length=max_race_laps,
    max_stops=2,
    supported_compounds={"SOFT", "MEDIUM", "HARD"},
    )
)

display(
    target_candidates[
        ["strategy_id", "pit_sequence", "pit_sequence_str",
         "num_stops", "is_baseline"]
    ].head(15)
)

print("\nStay Out candidate:")
display(target_candidates[target_candidates["num_stops"] == 0])

NameError: name 'target_driver' is not defined

In [ ]:
for sid in ["S001", "S002", "S038"]:
    if sid in own_replay_results:
        print(
            sid,
            own_replay_results[sid].summary["final_cumulative_time"],
            own_replay_results[sid].summary["final_position"]
        )
    else:
        print(sid, "NOT FOUND")

print(
    strategy_evaluation[
        strategy_evaluation["strategy_id"].isin(["S001", "S002", "S038"])
    ]
)

S001 5246.646658100015 17
S002 5241.799956656418 17
S038 5151.090226306743 17
    strategy_rank strategy_id  projected_finish_time_s  win_probability  \
0               1        S038              5151.090226              0.0   
79             80        S002              5241.799957              0.0   
84             85        S001              5246.646658              0.0   

    podium_probability  top5_probability  expected_finish_position  \
0                 0.01             0.158                     7.345   
79                0.00             0.000                    13.532   
84                0.00             0.000                    13.786   

    median_finish_position  worst_decile_position  scenarios_evaluated  \
0                      7.0                   10.0                 1000   
79                    14.0                   15.0                 1000   
84                    14.0                   15.0                 1000   

    rivals_included  
0                18  

In [ ]:
# Compare actual and projected timing components

target_rows = df[
    (df["driver_number"] == 44)
    & (df["lap_number"] <= 26)
].sort_values("lap_number")

target_elapsed = target_rows["lap_duration"].sum()

print("Target elapsed time at lap 26:", target_elapsed)

for sid in ["S001", "S002", "S038"]:
    result = own_replay_results[sid]
    summary = result.summary

    print(f"\n{sid}")
    print("Replay final position:", summary["final_position"])
    print("Replay cumulative time:", summary["final_cumulative_time"])
    print("Time after decision lap:",
          summary["final_cumulative_time"] - target_elapsed)

    print("Lap table columns:", result.laps_table.columns.tolist())
    print("Last target rows:")
    display(result.laps_table.tail(3))

Target elapsed time at lap 26: 2425.794

S001
Replay final position: 17
Replay cumulative time: 5246.646658100015
Time after decision lap: 2820.852658100015
Lap table columns: ['lap_number', 'driver_number', 'lap_duration', 'cum_time', 'position', 'gap_to_leader', 'gap_ahead', 'gap_behind', 'compound', 'tyre_age', 'is_target', 'ground_truth_available']
Last target rows:


,lap_number,driver_number,lap_duration,cum_time,position,gap_to_leader,gap_ahead,gap_behind,compound,tyre_age,is_target,ground_truth_available
1010,52,16,96.784272,5252.154989,18,2526.624049,5.508331,22.383796,INTERMEDIATE,32.0,False,False
1011,52,31,101.965676,5274.538785,19,2549.007844,22.383796,83.162956,MEDIUM,30.0,False,False
1012,52,24,98.257647,5357.701741,20,2632.170801,83.162956,0.000000,INTERMEDIATE,32.0,False,False



S002
Replay final position: 17
Replay cumulative time: 5241.799956656418
Time after decision lap: 2816.0059566564178
Lap table columns: ['lap_number', 'driver_number', 'lap_duration', 'cum_time', 'position', 'gap_to_leader', 'gap_ahead', 'gap_behind', 'compound', 'tyre_age', 'is_target', 'ground_truth_available']
Last target rows:


,lap_number,driver_number,lap_duration,cum_time,position,gap_to_leader,gap_ahead,gap_behind,compound,tyre_age,is_target,ground_truth_available
1010,52,16,96.784272,5252.154989,18,2526.624049,10.355032,22.383796,INTERMEDIATE,32.0,False,False
1011,52,31,101.965676,5274.538785,19,2549.007844,22.383796,83.162956,MEDIUM,30.0,False,False
1012,52,24,98.257647,5357.701741,20,2632.170801,83.162956,0.000000,INTERMEDIATE,32.0,False,False



S038
Replay final position: 17
Replay cumulative time: 5151.090226306743
Time after decision lap: 2725.296226306743
Lap table columns: ['lap_number', 'driver_number', 'lap_duration', 'cum_time', 'position', 'gap_to_leader', 'gap_ahead', 'gap_behind', 'compound', 'tyre_age', 'is_target', 'ground_truth_available']
Last target rows:


,lap_number,driver_number,lap_duration,cum_time,position,gap_to_leader,gap_ahead,gap_behind,compound,tyre_age,is_target,ground_truth_available
1010,52,16,96.784272,5252.154989,18,2526.624049,101.064763,22.383796,INTERMEDIATE,32.0,False,False
1011,52,31,101.965676,5274.538785,19,2549.007844,22.383796,83.162956,MEDIUM,30.0,False,False
1012,52,24,98.257647,5357.701741,20,2632.170801,83.162956,0.000000,INTERMEDIATE,32.0,False,False


In [ ]:
for sid in ["S001", "S002", "S038"]:

    result = own_replay_results[sid]
    laps = result.laps_table

    print(f"\n{'=' * 45}")
    print(f"STRATEGY: {sid}")

    print("\nTarget driver's final row:")
    display(
        laps[
            (laps["driver_number"] == 44)
            & (laps["lap_number"] == 52)
        ]
    )

    print("\nFinal classification:")
    display(
        laps[laps["lap_number"] == 52]
        .sort_values("cum_time")[
            ["driver_number", "cum_time", "position", "is_target"]
        ]
    )


STRATEGY: S001

Target driver's final row:


,lap_number,driver_number,lap_duration,cum_time,position,gap_to_leader,gap_ahead,gap_behind,compound,tyre_age,is_target,ground_truth_available
1009,52,44,103.47745,5246.646658,17,2521.115718,98.189996,5.508331,SOFT,14.0,True,True



Final classification:


,driver_number,cum_time,position,is_target
993,10,2725.530940,1,False
994,4,4978.248487,2,False
995,81,4979.912099,3,False
996,63,4989.533341,4,False
997,1,5000.593050,5,False
998,55,5008.137391,6,False
999,27,5037.437716,7,False
1000,14,5042.501302,8,False
1001,18,5044.433212,9,False
1002,23,5047.513867,10,False



STRATEGY: S002

Target driver's final row:


,lap_number,driver_number,lap_duration,cum_time,position,gap_to_leader,gap_ahead,gap_behind,compound,tyre_age,is_target,ground_truth_available
1009,52,44,103.424342,5241.799957,17,2516.269016,93.343295,10.355032,SOFT,15.0,True,True


... (2 outputs left out)


STRATEGY: S038

Target driver's final row:


,lap_number,driver_number,lap_duration,cum_time,position,gap_to_leader,gap_ahead,gap_behind,compound,tyre_age,is_target,ground_truth_available
1009,52,44,103.583666,5151.090226,17,2425.559286,2.633565,101.064763,SOFT,12.0,True,True



Final classification:


,driver_number,cum_time,position,is_target
993,10,2725.530940,1,False
994,4,4978.248487,2,False
995,81,4979.912099,3,False
996,63,4989.533341,4,False
997,1,5000.593050,5,False
998,55,5008.137391,6,False
999,27,5037.437716,7,False
1000,14,5042.501302,8,False
1001,18,5044.433212,9,False
1002,23,5047.513867,10,False


In [ ]:
for sid in ["S001", "S002", "S038"]:

    result = own_replay_results[sid]
    laps = result.laps_table

    print(f"\n{'=' * 45}")
    print(f"STRATEGY: {sid}")

    print("\nTarget driver's final row:")
    display(
        laps[
            (laps["driver_number"] == 44)
            & (laps["lap_number"] == 52)
        ]
    )

    print("\nFinal classification:")
    display(
        laps[laps["lap_number"] == 52]
        .sort_values("cum_time")[
            ["driver_number", "cum_time", "position", "is_target"]
        ]
    )


STRATEGY: S001

Target driver's final row:


,lap_number,driver_number,lap_duration,cum_time,position,gap_to_leader,gap_ahead,gap_behind,compound,tyre_age,is_target,ground_truth_available
1009,52,44,103.47745,5246.646658,17,2521.115718,98.189996,5.508331,SOFT,14.0,True,True



Final classification:


,driver_number,cum_time,position,is_target
993,10,2725.530940,1,False
994,4,4978.248487,2,False
995,81,4979.912099,3,False
996,63,4989.533341,4,False
997,1,5000.593050,5,False
998,55,5008.137391,6,False
999,27,5037.437716,7,False
1000,14,5042.501302,8,False
1001,18,5044.433212,9,False
1002,23,5047.513867,10,False



STRATEGY: S002

Target driver's final row:


,lap_number,driver_number,lap_duration,cum_time,position,gap_to_leader,gap_ahead,gap_behind,compound,tyre_age,is_target,ground_truth_available
1009,52,44,103.424342,5241.799957,17,2516.269016,93.343295,10.355032,SOFT,15.0,True,True


... (2 outputs left out)


STRATEGY: S038

Target driver's final row:


,lap_number,driver_number,lap_duration,cum_time,position,gap_to_leader,gap_ahead,gap_behind,compound,tyre_age,is_target,ground_truth_available
1009,52,44,103.583666,5151.090226,17,2425.559286,2.633565,101.064763,SOFT,12.0,True,True



Final classification:


,driver_number,cum_time,position,is_target
993,10,2725.530940,1,False
994,4,4978.248487,2,False
995,81,4979.912099,3,False
996,63,4989.533341,4,False
997,1,5000.593050,5,False
998,55,5008.137391,6,False
999,27,5037.437716,7,False
1000,14,5042.501302,8,False
1001,18,5044.433212,9,False
1002,23,5047.513867,10,False


In [ ]:
laps = own_replay_results["S038"].laps_table

for driver in [10, 44, 4, 81]:
    driver_laps = (
        laps[laps["driver_number"] == driver]
        .sort_values("lap_number")
    )

    print(f"\nDriver #{driver}")
    print("Rows:", len(driver_laps))
    print("First lap:", driver_laps["lap_number"].min())
    print("Last lap:", driver_laps["lap_number"].max())
    print("First cumulative time:", driver_laps["cum_time"].iloc[0])
    print("Last cumulative time:", driver_laps["cum_time"].iloc[-1])

    display(
        driver_laps[
            ["lap_number", "lap_duration", "cum_time", "position"]
        ].head(5)
    )


Driver #10
Rows: 26
First lap: 27
Last lap: 52
First cumulative time: 104.82811308928889
Last cumulative time: 2725.5309403215115


,lap_number,lap_duration,cum_time,position
493,27,104.828113,104.828113,1
513,28,104.828113,209.656226,1
533,29,104.828113,314.484339,1
553,30,104.828113,419.312452,1
573,31,104.828113,524.140565,1



Driver #44
Rows: 52
First lap: 1
Last lap: 52
First cumulative time: 96.402
Last cumulative time: 5151.090226306743


,lap_number,lap_duration,cum_time,position
1,1,96.402,96.402,2
20,2,91.420,187.822,3
39,3,91.716,279.538,3
58,4,91.988,371.526,3
77,5,91.677,463.203,3



Driver #4
Rows: 52
First lap: 1
Last lap: 52
First cumulative time: 97.721
Last cumulative time: 4978.248487294431


,lap_number,lap_duration,cum_time,position
3,1,97.721,97.721,4
22,2,92.188,189.909,5
41,3,91.991,281.900,5
60,4,91.976,373.876,5
79,5,91.716,465.592,5



Driver #81
Rows: 52
First lap: 1
Last lap: 52
First cumulative time: 98.359
Last cumulative time: 4979.912098568204


,lap_number,lap_duration,cum_time,position
4,1,98.359,98.359,5
23,2,92.361,190.720,6
42,3,91.927,282.647,6
61,4,91.919,374.566,6
80,5,91.832,466.398,6


In [ ]:
import inspect
import pitwall.engine as engine

print(inspect.getsource(engine.replay))

d10 = df[df["driver_number"] == 10]

print("Rows:", len(d10))
print("Lap range:", d10["lap_number"].min(), d10["lap_number"].max())
print("Unique laps:", sorted(d10["lap_number"].unique()))

def replay(
    session_df: pd.DataFrame,
    target_driver: int,
    decision_lap: int,
    strategy: Optional[Action] = None,
    pace_model=None,
    pit_loss: float = 22.0,
    track_mode: str = "replay",
) -> Union[ReplayResult, Dict[str, ReplayResult]]:
    """
    Replay a race with a counterfactual strategy for the target driver.

    Modes:
    - replay: Uses actual future rival lap times and historical lap effects.
    - causal: Forecasts target and rival pace without using actual future
              lap times or future lap effects.

    Important:
    Causal rival forecasts assume rivals do not make future pit stops.
    """

    # --------------------------------------------------------
    # 1. INPUT VALIDATION
    # --------------------------------------------------------

    if session_df is None or session_df.empty:
        raise ValueError("session_df must contain race-lap data.")

    required = {
        "driver_number",
        "lap_number",
        "lap_duration"

In [ ]:
import importlib
import pitwall.engine as engine
from pitwall.engine import Stay

importlib.invalidate_caches()
engine = importlib.reload(engine)

print(engine.__file__)

from pitwall.engine import Strategy, replay

test_result = replay(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    strategy=Strategy(
        pit_sequence=(),
        description="Stay Out"
    ),
    pace_model=model,
    pit_loss=22.0,
    track_mode="causal",
)

print(test_result.summary)

final_classification = (
    test_result.laps_table[
        test_result.laps_table["lap_number"] == 52
    ]
    .sort_values("position")
)

display(
    final_classification[
        ["driver_number", "cum_time", "position"]
    ]
)

C:\Users\Nihan\.vscode\pitwall\pitwall\engine.py
{'target_driver': 44, 'decision_lap': 26, 'ground_truth_available': True, 'final_position': 3, 'final_cumulative_time': 4980.937245905431, 'total_time_delta_vs_actual': 33.510245905430565, 'positions_gained_vs_actual': -2}


,driver_number,cum_time,position
968,4,4978.248487,1
969,81,4979.912099,2
970,44,4980.937246,3
971,63,4989.533341,4
972,1,5000.593050,5
973,55,5008.137391,6
974,27,5037.437716,7
975,14,5042.501302,8
976,18,5044.433212,9
977,23,5047.513867,10


In [ ]:
for driver in [3, 11, 16, 24, 31, 63, 77]:
    driver_df = (
        df[
            df["driver_number"] == driver
        ].sort_values("lap_number")
    )

    print(
        f"Driver {driver}: "
        f"last lap={driver_df['lap_number'].max()}, "
        f"rows={len(driver_df)}"
    )

Driver 3: last lap=51, rows=51
Driver 11: last lap=50, rows=50
Driver 16: last lap=51, rows=51
Driver 24: last lap=50, rows=50
Driver 31: last lap=50, rows=50
Driver 63: last lap=34, rows=34
Driver 77: last lap=51, rows=51


In [ ]:
import importlib
import pitwall.strategy_generator as sg

importlib.invalidate_caches()
sg = importlib.reload(sg)

candidates = sg.generate_multi_stop_candidates(
    race_df=df,
    target_driver=44,
    decision_lap=26,
    race_length=52,
    max_stops=2,
    supported_compounds={"SOFT", "MEDIUM", "HARD"},
)

print("Candidate count:", len(candidates))
display(candidates.head(15))

assert "is_baseline" not in candidates.columns
assert "baseline_distance" not in candidates.columns

assert all(
    lap > 26
    for sequence in candidates["pit_sequence"]
    for lap, compound in sequence
)

print("Decision-time candidate validation passed.")

Candidate count: 67


,pit_sequence,pit_sequence_str,current_compound,num_stops,description,strategy_id
0,(),Stay out,MEDIUM,0,Stay out,S001
1,"((31, HARD),)",HARD lap 31,MEDIUM,1,"Pit lap 31, fit HARD",S002
2,"((36, HARD),)",HARD lap 36,MEDIUM,1,"Pit lap 36, fit HARD",S003
3,"((41, HARD),)",HARD lap 41,MEDIUM,1,"Pit lap 41, fit HARD",S004
4,"((46, HARD),)",HARD lap 46,MEDIUM,1,"Pit lap 46, fit HARD",S005
5,"((31, MEDIUM),)",MEDIUM lap 31,MEDIUM,1,"Pit lap 31, fit MEDIUM",S006
6,"((36, MEDIUM),)",MEDIUM lap 36,MEDIUM,1,"Pit lap 36, fit MEDIUM",S007
7,"((41, MEDIUM),)",MEDIUM lap 41,MEDIUM,1,"Pit lap 41, fit MEDIUM",S008
8,"((46, MEDIUM),)",MEDIUM lap 46,MEDIUM,1,"Pit lap 46, fit MEDIUM",S009
9,"((31, SOFT),)",SOFT lap 31,MEDIUM,1,"Pit lap 31, fit SOFT",S010


Decision-time candidate validation passed.


In [ ]:

from pandas.testing import assert_frame_equal

decision_lap = 26

# Original candidate generation
original_candidates = generate_multi_stop_candidates(
    race_df=df,
    target_driver=44,
    decision_lap=decision_lap,
    race_length=52,
    supported_compounds={"SOFT", "MEDIUM", "HARD"},
)

# Create a copy with future data deliberately corrupted
mutated_df = df.copy()

future_mask = mutated_df["lap_number"] > decision_lap

# Alter future values
mutated_df.loc[future_mask, "lap_duration"] = 9999
mutated_df.loc[future_mask, "compound"] = "CORRUPTED"
mutated_df.loc[future_mask, "tyre_age"] = -999
mutated_df.loc[future_mask, "rainfall"] = 999

# Generate candidates again
mutated_candidates = generate_multi_stop_candidates(
    race_df=mutated_df,
    target_driver=44,
    decision_lap=decision_lap,
    race_length=52,
    supported_compounds={"SOFT", "MEDIUM", "HARD"},
)

# Compare outputs
assert_frame_equal(
    original_candidates.reset_index(drop=True),
    mutated_candidates.reset_index(drop=True),
)

print("PASS: Candidate generation is independent of future race data.")


PASS: Candidate generation is independent of future race data.


In [ ]:

import importlib
import pitwall.engine

importlib.reload(pitwall.engine)

from pitwall.engine import replay, Strategy

print("Updated replay signature:")
import inspect
print(inspect.signature(replay))


test_result = replay(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    strategy=Strategy(
        pit_sequence=(),
        description="Stay Out"
    ),
    pace_model=model,
    pit_loss=22.0,
    track_mode="causal",
    race_length=52,
)

print(test_result.summary)



Updated replay signature:
(session_df: pandas.DataFrame, target_driver: int, decision_lap: int, strategy: pitwall.engine.Stay | pitwall.engine.Pit | pitwall.engine.Strategy | None = None, pace_model=None, pit_loss: float = 22.0, track_mode: str = 'replay', race_length: int | None = None) -> pitwall.engine.ReplayResult | Dict[str, pitwall.engine.ReplayResult]
{'target_driver': 44, 'decision_lap': 26, 'race_length': 52, 'ground_truth_available': True, 'final_position': 3, 'final_cumulative_time': 4980.937245905431, 'total_time_delta_vs_actual': 33.510245905431475, 'positions_gained_vs_actual': -2}


In [ ]:

from pandas.testing import assert_frame_equal
from pitwall.engine import replay, Strategy

# Original causal replay
original = replay(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    strategy=Strategy(pit_sequence=(), description="Stay Out"),
    pace_model=model,
    track_mode="causal",
    race_length=52,
)

# Corrupt all observations after the decision lap
mutated_df = df.copy()
future_mask = mutated_df["lap_number"] > 26

mutated_df.loc[future_mask, "lap_duration"] = 9999
mutated_df.loc[future_mask, "compound"] = "CORRUPTED"
mutated_df.loc[future_mask, "tyre_age"] = -999

# Repeat causal replay
mutated = replay(
    session_df=mutated_df,
    target_driver=44,
    decision_lap=26,
    strategy=Strategy(pit_sequence=(), description="Stay Out"),
    pace_model=model,
    track_mode="causal",
    race_length=52,
)

# Compare projected target lap times and positions
original_target = original.laps_table[
    (original.laps_table["driver_number"] == 44)
    & (original.laps_table["lap_number"] > 26)
].reset_index(drop=True)

mutated_target = mutated.laps_table[
    (mutated.laps_table["driver_number"] == 44)
    & (mutated.laps_table["lap_number"] > 26)
].reset_index(drop=True)

assert_frame_equal(original_target, mutated_target)

print("PASS: Target causal replay is independent of future observations.")


PASS: Target causal replay is independent of future observations.


In [ ]:
from pitwall.rival_state import build_rival_state

rival_state = build_rival_state(
    session_df=df,
    target_driver=44,
    decision_lap=26,
)

print("Rival state shape:", rival_state.shape)
display(rival_state.head())

print("rival_strategy_probs:", "rival_strategy_probs" in globals())

Rival state shape: (19, 12)


,driver_number,lap_observed,laps_behind_target,position,compound,tyre_age,stint_number,pit_stops_observed,stint_start_lap,stint_length,last_pit_lap,gap_to_target_s
0,4,26,0,1.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,-2.861
1,81,26,0,2.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,-2.279
2,63,26,0,4.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,1.342
3,1,26,0,5.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,9.478
4,55,26,0,6.0,MEDIUM,25.0,1.0,0.0,1.0,26.0,NaN,10.661


rival_strategy_probs: False


In [ ]:
from pitwall.rival_strategy import generate_rival_strategies
from pitwall.rival_probability import assign_rival_strategy_probabilities

rival_strategies = generate_rival_strategies(
    rival_state=rival_state,
    decision_lap=26,
    race_end_lap=52,
)

rival_strategy_probs = assign_rival_strategy_probabilities(
    rival_strategies
)

print("Rival strategies:", len(rival_strategies))

Rival strategies: 1800


In [ ]:
from pitwall.rival_scenario import simulate_rival_scenarios

rival_scenario_results = simulate_rival_scenarios(
    rival_state=rival_state,
    rival_strategy_probs=rival_strategy_probs,
    pace_model=model,
    decision_lap=26,
    race_end_lap=52,
    pit_loss=22.0,
)

print("Rival scenarios generated:", len(rival_scenario_results))

rival_scenario_results.head()

print("rival_state:", "rival_state" in globals())
print("rival_strategy_probs:", "rival_strategy_probs" in globals())

Rival scenarios generated: 2


AttributeError: 'tuple' object has no attribute 'head'

In [ ]:
rival_scenario_results, rival_scenario_summary = simulate_rival_scenarios(
    rival_state=rival_state,
    rival_strategy_probs=rival_strategy_probs,
    pace_model=model,
    decision_lap=26,
    race_end_lap=52,
    pit_loss=22.0,
)

print("Scenario results:", rival_scenario_results.shape)
print("Scenario summary:", rival_scenario_summary.shape)

display(rival_scenario_results.head())
display(rival_scenario_summary.head())

Scenario results: (1800, 10)
Scenario summary: (18, 6)


,strategy_id,driver_number,additional_stops,pit_laps,compounds_after_stops,strategy_probability,projected_future_time_s,projected_pit_loss_s,projected_total_time_s,projected_laps
0,D4_S001,4,0,(),(),0.150000,2555.315487,0.0,2555.315487,"[{'lap_number': 27, 'compound': 'MEDIUM', 'tyr..."
1,D4_S002,4,1,"(26,)","(SOFT,)",0.040625,2713.276337,22.0,2713.276337,"[{'lap_number': 27, 'compound': 'SOFT', 'tyre_..."
2,D4_S003,4,1,"(26,)","(HARD,)",0.040625,2688.858438,22.0,2688.858438,"[{'lap_number': 27, 'compound': 'HARD', 'tyre_..."
3,D4_S004,4,1,"(29,)","(SOFT,)",0.040625,2704.942494,22.0,2704.942494,"[{'lap_number': 27, 'compound': 'MEDIUM', 'tyr..."
4,D4_S005,4,1,"(29,)","(HARD,)",0.040625,2685.773787,22.0,2685.773787,"[{'lap_number': 27, 'compound': 'MEDIUM', 'tyr..."


,driver_number,scenario_count,expected_future_time_s,best_case_time_s,worst_case_time_s,most_likely_strategy
0,1,100,2669.450125,2565.321050,2754.204052,D1_S001
1,2,100,2698.071103,2593.942029,2782.825031,D2_S001
2,3,100,2713.463872,2609.334797,2798.217800,D3_S001
3,4,100,2659.444562,2555.315487,2744.198490,D4_S001
4,11,100,2758.392599,2703.678210,2811.322917,D11_S001


In [ ]:
evaluation = evaluate_strategies_against_rivals(
    own_replay_results=own_replay_results,
    rival_scenario_results=rival_scenario_results,
    rival_state=rival_state,
    session_df=df,
    target_driver=44,
    decision_lap=26,
    n_scenarios=1000,
    random_state=42,
)

display(evaluation.head(15))

,strategy_rank,strategy_id,projected_finish_time_s,win_probability,podium_probability,top5_probability,expected_finish_position,median_finish_position,worst_decile_position,scenarios_evaluated,rivals_included
0,1,S001,4980.937246,0.733,1.000,1.000,1.296,1.0,2.0,1000,18
1,2,S005,5045.696517,0.218,0.860,0.996,2.384,2.0,4.0,1000,18
2,3,S009,5046.150703,0.218,0.860,0.996,2.384,2.0,4.0,1000,18
3,4,S013,5047.102300,0.218,0.860,0.996,2.384,2.0,4.0,1000,18
4,5,S008,5073.359176,0.060,0.560,0.905,3.494,3.0,5.0,1000,18
5,6,S004,5073.529056,0.060,0.560,0.905,3.494,3.0,5.0,1000,18
6,7,S012,5078.044670,0.054,0.523,0.890,3.618,3.0,6.0,1000,18
7,8,S007,5092.565157,0.031,0.392,0.819,4.094,4.0,6.0,1000,18
8,9,S003,5094.270515,0.027,0.376,0.808,4.173,4.0,6.0,1000,18
9,10,S029,5099.236856,0.015,0.312,0.756,4.447,4.0,6.1,1000,18


In [ ]:
for strategy_id in ["S001", "S005"]:
    result = own_replay_results[strategy_id]

    print(f"\n{strategy_id}")
    print(result.summary)

    display(
        result.laps_table[
            result.laps_table["lap_number"] > 26
        ][
            ["lap_number", "lap_duration", "compound", "tyre_age"]
        ].head(15)
    )


S001
{'target_driver': 44, 'decision_lap': 26, 'race_length': 52, 'ground_truth_available': True, 'final_position': 3, 'final_cumulative_time': 4980.937245905431, 'total_time_delta_vs_actual': 33.510245905431475, 'positions_gained_vs_actual': -2}


,lap_number,lap_duration,compound,tyre_age
493,27,100.281988,MEDIUM,26
494,27,100.323588,MEDIUM,26
495,27,100.275363,MEDIUM,26
496,27,100.554367,MEDIUM,26
497,27,100.666817,MEDIUM,26
498,27,100.911484,MEDIUM,26
499,27,111.469522,INTERMEDIATE,7
500,27,101.388881,MEDIUM,26
501,27,101.577631,MEDIUM,26
502,27,101.436365,MEDIUM,26



S005
{'target_driver': 44, 'decision_lap': 26, 'race_length': 52, 'ground_truth_available': True, 'final_position': 9, 'final_cumulative_time': 5045.696517026345, 'total_time_delta_vs_actual': 98.26951702634506, 'positions_gained_vs_actual': -8}


,lap_number,lap_duration,compound,tyre_age
493,27,100.281988,MEDIUM,26
494,27,100.323588,MEDIUM,26
495,27,100.275363,MEDIUM,26
496,27,100.554367,MEDIUM,26
497,27,100.666817,MEDIUM,26
498,27,100.911484,MEDIUM,26
499,27,111.469522,INTERMEDIATE,7
500,27,101.388881,MEDIUM,26
501,27,101.577631,MEDIUM,26
502,27,101.436365,MEDIUM,26


In [ ]:
for strategy_id in ["S001", "S005"]:
    result = own_replay_results[strategy_id]

    target_laps = result.laps_table[
        (result.laps_table["driver_number"] == 44) &
        (result.laps_table["lap_number"] > 26)
    ].copy()

    print(f"\n{strategy_id}")
    print("Future laps:", len(target_laps))

    display(
        target_laps[
            ["lap_number", "lap_duration", "compound", "tyre_age"]
        ].head(30)
    )

for strategy_id in ["S001", "S005"]:
    result = own_replay_results[strategy_id]

    target_laps = result.laps_table[
        result.laps_table["driver_number"] == 44
    ].copy()

    print(f"\n{strategy_id}")
    print("Pit-stop sequence:", result.strategy.pit_sequence
          if hasattr(result, "strategy") else "Inspecting result attributes")

    print(
        target_laps[
            target_laps["lap_number"] > 26
        ]["lap_duration"].describe()
    )


S001
Future laps: 26


,lap_number,lap_duration,compound,tyre_age
495,27,100.275363,MEDIUM,26
514,28,100.115313,MEDIUM,27
533,29,99.955264,MEDIUM,28
552,30,99.795214,MEDIUM,29
571,31,99.635164,MEDIUM,30
590,32,99.475114,MEDIUM,31
609,33,99.315064,MEDIUM,32
628,34,99.155014,MEDIUM,33
647,35,98.994965,MEDIUM,34
666,36,98.834915,MEDIUM,35



S005
Future laps: 26


,lap_number,lap_duration,compound,tyre_age
495,27,100.275363,MEDIUM,26
514,28,100.115313,MEDIUM,27
533,29,99.955264,MEDIUM,28
552,30,99.795214,MEDIUM,29
571,31,99.635164,MEDIUM,30
590,32,99.475114,MEDIUM,31
609,33,99.315064,MEDIUM,32
628,34,99.155014,MEDIUM,33
647,35,98.994965,MEDIUM,34
666,36,98.834915,MEDIUM,35



S001
Pit-stop sequence: Inspecting result attributes
count     26.000000
mean      98.274740
std        1.224146
min       96.274117
25%       97.274429
50%       98.274740
75%       99.275052
max      100.275363
Name: lap_duration, dtype: float64

S005
Pit-stop sequence: Inspecting result attributes
count     26.000000
mean     100.765481
std        3.956367
min       97.394466
25%       98.394778
50%       99.395089
75%      102.687694
max      115.109770
Name: lap_duration, dtype: float64


In [ ]:
print(model.__class__)
print(vars(model).keys())

import inspect
print(inspect.getsource(model.predict_lap))

for age in [1, 5, 10, 15, 20, 26, 30, 40, 50]:
    pace = model.predict_lap(
        lap_number=40,
        driver_number=44,
        compound="MEDIUM",
        tyre_age=age,
        mode="causal",
        decision_lap=26,
    )
    print(f"Tyre age {age:2}: {pace:.3f}s")

model.compound_age_slopes

print("Compound age slopes:")
print(model.compound_age_slopes)

print("\nCompound effects:")
print(model.compound_effects)

print("\nCoefficients:")
print(model.coefficients)

print("\nIdentification report:")
print(model.identification_report)

<class 'pitwall.pace_model.PaceModel'>
dict_keys(['reference_compound', 'fitted', 'intercept', 'lap_effects', 'driver_effects', 'compound_effects', 'compound_age_slopes', 'coefficients', 'identification_report', 'rsquared', 'base_driver', 'base_lap'])
    def predict_lap(
        self,
        lap_number: int,
        driver_number: int,
        compound: str,
        tyre_age: int,
        mode: str = "replay",
        decision_lap: Optional[int] = None,
    ) -> float:
        """Predict expected clean lap time for a driver on a lap.
        
        Modes:
        - 'replay': Uses actual field lap effects for historical replay.
        - 'causal': Holds the decision-lap effect constant for future predictions.
        """
        if not self.fitted:
            raise ValueError("Model must be fitted before predicting.")

        if mode == "replay":
            lap_eff = self.lap_effects.get(lap_number, self.lap_effects.get(self.base_lap, 0.0))
        elif mode == "causal":
        

In [ ]:
from pitwall.engine import replay

own_replay_results = replay(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    strategy=None,
    pace_model=model,
    pit_loss=22.0,
    track_mode="causal",
    race_length=52,
)

print(f"Strategies generated: {len(own_replay_results)}")

Strategies generated: 67


In [ ]:
# Inspect existing dataframes in the notebook
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        relevant = [
            c for c in obj.columns
            if any(x in c.lower() for x in [
                "fuel", "track_evolution", "grip"
            ])
        ]

        if relevant:
            print(f"\n{name}: {obj.shape}")
            print(relevant)


df: (962, 32)
['grip_condition']

mutated_df: (962, 32)
['grip_condition']

pace_data: (726, 32)
['grip_condition']


In [ ]:
pace_data = df.copy()

pace_data = pace_data[
    pace_data["compound"].isin(["SOFT", "MEDIUM", "HARD"])
]

age_summary = (
    pace_data.groupby(["compound", "tyre_age"])["lap_duration"]
    .agg(["count", "mean", "median", "std"])
    .round(3)
)

display(age_summary)

import inspect
from pitwall.pace_model import PaceModel

print(inspect.getsource(PaceModel.fit))

count     mean   median    std
compound tyre_age                                
HARD     0.0           2  117.532  117.532  1.102
         1.0           3   91.158   90.475  1.559
         2.0           3   90.605   89.872  1.508
         3.0           3   90.638   89.872  1.685
         4.0           3   90.318   89.739  1.585
...                  ...      ...      ...    ...
SOFT     14.0          2   93.433   93.433  2.052
         15.0          1   96.522   96.522    NaN
         16.0          1   98.735   98.735    NaN
         17.0          1   98.497   98.497    NaN
         18.0          1  101.731  101.731    NaN

[66 rows x 4 columns]

    def fit(self, df: pd.DataFrame) -> "PaceModel":
        """Fit the model on clean laps only."""
        clean_df = df[df["is_clean_lap"]].copy()

        # Categorical ordering with reference compound first
        categories = [self.reference_compound] + [
            c for c in ["HARD", "INTERMEDIATE", "MEDIUM", "SOFT"] if c != self.reference_compound
        ]
        clean_df["compound"] = pd.Categorical(clean_df["compound"], categories=categories)

        # Identification report
        compounds = ["MEDIUM", "HARD", "INTERMEDIATE", "SOFT"]
        pairs = list(combinations(compounds, 2))
        pair_counts = {}
        weak_pairs = []

        for p in pairs:
            p_name = f"{p[0]} vs {p[1]}"
            cnt = 0
            for _, grp in clean_df.groupby("lap_number"):
                pres = set(grp["compound"].unique())
                if p[0] in pres and p[1] in pres:
                    cnt += 1
            pair_counts[p_name] = cnt
            if cnt < 15:
      

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. Prepare data
# --------------------------------------------------

data = df.copy()

data = data[
    data["is_clean_lap"]
    & data["compound"].isin(["SOFT", "MEDIUM", "HARD"])
].copy()

data = data.sort_values("lap_number")

features = [
    "lap_number",
    "driver_number",
    "compound",
    "tyre_age",
    "track_temperature",
    "grip_condition",
    "rainfall",
    "air_temperature",
    "wind_speed",
]

target = "lap_duration"

data = data.dropna(subset=[target, "lap_number", "driver_number",
                           "compound", "tyre_age"])

# --------------------------------------------------
# 2. Model specifications
# --------------------------------------------------

model_features = {
    "M0_Baseline": [
        "lap_number", "driver_number",
        "compound", "tyre_age"
    ],

    "M1_Track_Adjusted": [
        "lap_number", "driver_number",
        "compound", "tyre_age",
        "track_temperature", "grip_condition",
        "rainfall", "air_temperature", "wind_speed"
    ],

    "M2_Trend_Adjusted": [
        "lap_number", "driver_number",
        "compound", "tyre_age",
        "track_temperature", "grip_condition",
        "rainfall", "air_temperature", "wind_speed"
    ],
}

categorical = ["driver_number", "compound", "grip_condition"]

folds = [
    (15, 24),
    (24, 32),
    (32, 40),
    (40, 46),
    (46, 52),
]

results = []

for model_name, cols in model_features.items():

    for train_end, test_end in folds:

        train = data[data["lap_number"] <= train_end].copy()

        test = data[
            (data["lap_number"] > train_end)
            & (data["lap_number"] <= test_end)
        ].copy()

        if train.empty or test.empty:
            continue

        cats = [c for c in categorical if c in cols]

        numeric = [c for c in cols if c not in cats]

        if model_name == "M2_Trend_Adjusted":
            train["lap_squared"] = train["lap_number"] ** 2
            test["lap_squared"] = test["lap_number"] ** 2
            numeric.append("lap_squared")

        transformers = [
            ("numeric", StandardScaler(), numeric),
            ("categorical",
             OneHotEncoder(handle_unknown="ignore"),
             cats)
        ]

        preprocessing = ColumnTransformer(transformers)

        pipeline = Pipeline([
            ("preprocessor", preprocessing),
            ("model", Ridge(alpha=1.0))
        ])

        pipeline.fit(train[cols + (["lap_squared"]
                                   if "lap_squared" in numeric else [])],
                     train[target])

        prediction = pipeline.predict(
            test[cols + (["lap_squared"]
                         if "lap_squared" in numeric else [])]
        )

        results.append({
            "model": model_name,
            "train_laps": train_end,
            "test_laps": f"{train_end + 1}-{test_end}",
            "MAE": mean_absolute_error(test[target], prediction),
            "RMSE": np.sqrt(mean_squared_error(test[target], prediction)),
            "max_abs_prediction": np.max(np.abs(prediction)),
            "prediction_min": np.min(prediction),
            "prediction_max": np.max(prediction),
        })

results_df = pd.DataFrame(results)

display(results_df.round(4))

,model,train_laps,test_laps,MAE,RMSE,max_abs_prediction,prediction_min,prediction_max
0,M0_Baseline,15,16-24,2.9942,4.5674,93.8891,91.3461,93.8891
1,M0_Baseline,24,25-32,3.9747,5.8159,96.9242,94.2924,96.9242
2,M0_Baseline,32,33-40,13.3042,13.3761,106.9710,103.8639,106.9710
3,M0_Baseline,40,41-46,2.6152,2.8916,97.1709,90.5780,97.1709
4,M0_Baseline,46,47-52,1.2490,1.5768,94.0948,89.6391,94.0948
5,M1_Track_Adjusted,15,16-24,2.9192,4.5217,93.9581,91.4702,93.9581
6,M1_Track_Adjusted,24,25-32,5.3034,7.2994,96.0615,92.1602,96.0615
7,M1_Track_Adjusted,32,33-40,4.8202,5.0772,98.9589,94.9497,98.9589
8,M1_Track_Adjusted,40,41-46,3.6936,5.3322,102.5129,89.9485,102.5129
9,M1_Track_Adjusted,46,47-52,0.8114,1.3877,92.1340,88.3353,92.1340


In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ============================================================
# 1. PREPARE DATA
# ============================================================

df = df.copy()

df = df[
    (df["is_clean_lap"] == True)
    & (df["compound"].isin(["SOFT", "MEDIUM", "HARD"]))
].copy()

df["lap_number"] = pd.to_numeric(df["lap_number"], errors="coerce")
df["lap_duration"] = pd.to_numeric(df["lap_duration"], errors="coerce")
df["tyre_age"] = pd.to_numeric(df["tyre_age"], errors="coerce")

df = df.dropna(
    subset=["lap_number", "lap_duration", "tyre_age",
            "driver_number", "compound"]
).copy()

df["lap_squared"] = df["lap_number"] ** 2

print("Evaluation dataset:", df.shape)
print("Lap range:", df["lap_number"].min(), "-", df["lap_number"].max())
print("Compounds:", sorted(df["compound"].unique()))


# ============================================================
# 2. DEFINE MODEL CANDIDATES
# ============================================================

base_numeric = [
    "lap_number",
    "tyre_age"
]

environment_numeric = [
    "track_temperature",
    "rainfall",
    "air_temperature",
    "wind_speed"
]

base_categorical = [
    "driver_number",
    "compound"
]

environment_categorical = [
    "grip_condition"
]

model_features = {
    "M0": {
        "numeric": base_numeric,
        "categorical": base_categorical
    },

    "M1": {
        "numeric": base_numeric + environment_numeric,
        "categorical": base_categorical + environment_categorical
    },

    "M2": {
        "numeric": base_numeric + environment_numeric + ["lap_squared"],
        "categorical": base_categorical + environment_categorical
    }
}


# ============================================================
# 3. CHRONOLOGICAL VALIDATION FOLDS
# ============================================================

folds = [
    (15, 24),
    (24, 32),
    (32, 40),
    (40, 46),
    (46, 52)
]


# ============================================================
# 4. MODEL FACTORY
# ============================================================

def build_model(numeric_features, categorical_features):

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessing = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    return Pipeline([
        ("preprocessing", preprocessing),
        ("regressor", Ridge(alpha=1.0))
    ])


# ============================================================
# 5. RUN ALL FOLDS AND SAVE OUT-OF-FOLD PREDICTIONS
# ============================================================

fold_results = []
oof_predictions = []

for model_name, features in model_features.items():

    numeric_features = features["numeric"]
    categorical_features = features["categorical"]
    all_features = numeric_features + categorical_features

    for train_end, test_end in folds:

        train = df[df["lap_number"] <= train_end].copy()

        test = df[
            (df["lap_number"] > train_end)
            & (df["lap_number"] <= test_end)
        ].copy()

        if train.empty or test.empty:
            print(f"Skipping {model_name}: fold {train_end}-{test_end}")
            continue

        X_train = train[all_features].copy()
        X_test = test[all_features].copy()

        y_train = train["lap_duration"]
        y_test = test["lap_duration"].to_numpy()

        # Handle missing categorical values consistently
        for col in categorical_features:
            X_train[col] = X_train[col].fillna("UNKNOWN").astype(str)
            X_test[col] = X_test[col].fillna("UNKNOWN").astype(str)

        model = build_model(
            numeric_features,
            categorical_features
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)

        errors = predictions - y_test

        fold_mae = mean_absolute_error(y_test, predictions)
        fold_rmse = np.sqrt(mean_squared_error(y_test, predictions))

        fold_results.append({
            "model": model_name,
            "train_laps": f"<= {train_end}",
            "test_laps": f"{train_end + 1}-{test_end}",
            "train_rows": len(train),
            "test_rows": len(test),
            "MAE": fold_mae,
            "RMSE": fold_rmse,
            "max_abs_error": np.max(np.abs(errors))
        })

        oof_predictions.append(pd.DataFrame({
            "model": model_name,
            "lap_number": test["lap_number"].to_numpy(),
            "actual": y_test,
            "predicted": predictions,
            "error": errors,
            "abs_error": np.abs(errors)
        }))


# ============================================================
# 6. FOLD-LEVEL RESULTS
# ============================================================

fold_results_df = pd.DataFrame(fold_results)

print("\nFOLD-LEVEL RESULTS")
display(
    fold_results_df.round(4)
)


# ============================================================
# 7. POOLED OUT-OF-FOLD METRICS
# ============================================================

predictions_df = pd.concat(
    oof_predictions,
    ignore_index=True
)

pooled_results = []

for model_name, group in predictions_df.groupby("model"):

    actual = group["actual"]
    predicted = group["predicted"]

    pooled_results.append({
        "model": model_name,
        "observations": len(group),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "max_abs_error": group["abs_error"].max(),
        "mean_bias": group["error"].mean(),
        "min_prediction": predicted.min(),
        "max_prediction": predicted.max()
    })

pooled_results_df = pd.DataFrame(pooled_results)

print("\nPOOLED OUT-OF-FOLD RESULTS")
display(
    pooled_results_df.round(4)
)


# ============================================================
# 8. SANITY CHECK
# ============================================================

print("\nSANITY CHECK")

for model_name in model_features:

    fold_rows = fold_results_df[
        fold_results_df["model"] == model_name
    ]

    pooled_rows = predictions_df[
        predictions_df["model"] == model_name
    ]

    print(
        f"{model_name}: "
        f"fold test rows = {fold_rows['test_rows'].sum()}, "
        f"pooled prediction rows = {len(pooled_rows)}"
    )

    assert fold_rows["test_rows"].sum() == len(pooled_rows)

print("\nEvaluation complete.")

Evaluation dataset: (650, 33)
Lap range: 2 - 51
Compounds: ['HARD', 'MEDIUM', 'SOFT']

FOLD-LEVEL RESULTS


,model,train_laps,test_laps,train_rows,test_rows,MAE,RMSE,max_abs_error
0,M0,<= 15,16-24,264,149,2.9942,4.5674,15.1631
1,M0,<= 24,25-32,413,26,3.9746,5.8158,15.1401
2,M0,<= 32,33-40,439,20,13.3042,13.3760,15.0755
3,M0,<= 40,41-46,459,108,2.6152,2.8916,6.2959
4,M0,<= 46,47-52,567,83,1.2490,1.5768,7.7209
5,M1,<= 15,16-24,264,149,2.9192,4.5217,15.1076
6,M1,<= 24,25-32,413,26,5.3034,7.2994,17.5867
7,M1,<= 32,33-40,439,20,4.8202,5.0772,6.7635
8,M1,<= 40,41-46,459,108,3.6940,5.3324,11.4007
9,M1,<= 46,47-52,567,83,0.8113,1.3875,8.3569



POOLED OUT-OF-FOLD RESULTS


,model,observations,MAE,RMSE,max_abs_error,mean_bias,min_prediction,max_prediction
0,M0,386,3.1131,4.7408,15.1631,0.2102,89.6391,106.9710
1,M1,386,2.9418,4.6029,17.5867,-0.3187,88.3353,102.5132
2,M2,386,5.8162,11.2549,47.4981,2.4175,88.1852,139.2021



SANITY CHECK
M0: fold test rows = 386, pooled prediction rows = 386
M1: fold test rows = 386, pooled prediction rows = 386
M2: fold test rows = 386, pooled prediction rows = 386

Evaluation complete.


In [ ]:

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ============================================================
# 1. PREPARE DATA
# ============================================================

df = df.copy()

df = df[
    (df["is_clean_lap"] == True)
    & (df["compound"].isin(["SOFT", "MEDIUM", "HARD"]))
].copy()

for col in ["lap_number", "lap_duration", "tyre_age"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(
    subset=["lap_number", "lap_duration", "tyre_age",
            "driver_number", "compound"]
).copy()

df["lap_number"] = df["lap_number"].astype(int)

# ============================================================
# 2. DEFINE FEATURE GROUPS
# ============================================================

base_numeric = ["lap_number", "tyre_age"]
base_categorical = ["driver_number", "compound"]

track_features = ["track_temperature"]
grip_features = ["grip_condition"]

weather_numeric = [
    "rainfall",
    "air_temperature",
    "wind_speed"
]

model_configs = {
    "M0": {
        "numeric": base_numeric,
        "categorical": base_categorical
    },

    "M1a": {
        "numeric": base_numeric + track_features,
        "categorical": base_categorical
    },

    "M1b": {
        "numeric": base_numeric + track_features,
        "categorical": base_categorical + grip_features
    },

    "M1c": {
        "numeric": base_numeric + weather_numeric,
        "categorical": base_categorical
    },

    "M1d": {
        "numeric": base_numeric + track_features + weather_numeric,
        "categorical": base_categorical + grip_features
    }
}

# ============================================================
# 3. CHRONOLOGICAL FOLDS
# ============================================================

folds = [
    (15, 24),
    (24, 32),
    (32, 40),
    (40, 46),
    (46, 52)
]

# ============================================================
# 4. MODEL PIPELINE
# ============================================================

def build_model(numeric_features, categorical_features):

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessing = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    return Pipeline([
        ("preprocessing", preprocessing),
        ("regressor", Ridge(alpha=1.0))
    ])


# ============================================================
# 5. RUN ALL MODELS AND FOLDS
# ============================================================

fold_results = []
oof_predictions = []

for model_name, config in model_configs.items():

    numeric_features = config["numeric"]
    categorical_features = config["categorical"]

    features = numeric_features + categorical_features

    for train_end, test_end in folds:

        train = df[df["lap_number"] <= train_end].copy()

        test = df[
            (df["lap_number"] > train_end)
            & (df["lap_number"] <= test_end)
        ].copy()

        if train.empty or test.empty:
            continue

        X_train = train[features].copy()
        X_test = test[features].copy()

        for col in categorical_features:
            X_train[col] = X_train[col].fillna("UNKNOWN").astype(str)
            X_test[col] = X_test[col].fillna("UNKNOWN").astype(str)

        y_train = train["lap_duration"].to_numpy()
        y_test = test["lap_duration"].to_numpy()

        model = build_model(
            numeric_features,
            categorical_features
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        errors = predictions - y_test

        fold_results.append({
            "model": model_name,
            "train_laps": f"<= {train_end}",
            "test_laps": f"{train_end + 1}-{test_end}",
            "train_rows": len(train),
            "test_rows": len(test),
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
            "max_abs_error": np.max(np.abs(errors))
        })

        oof_predictions.append(pd.DataFrame({
            "model": model_name,
            "lap_number": test["lap_number"].to_numpy(),
            "actual": y_test,
            "predicted": predictions,
            "error": errors,
            "abs_error": np.abs(errors)
        }))


# ============================================================
# 6. FOLD-LEVEL RESULTS
# ============================================================

fold_results_df = pd.DataFrame(fold_results)

print("FOLD-LEVEL RESULTS")
display(fold_results_df.round(4))


# ============================================================
# 7. POOLED OUT-OF-FOLD RESULTS
# ============================================================

predictions_df = pd.concat(oof_predictions, ignore_index=True)

pooled_results = []

for model_name, group in predictions_df.groupby("model"):

    pooled_results.append({
        "model": model_name,
        "observations": len(group),
        "MAE": mean_absolute_error(
            group["actual"], group["predicted"]
        ),
        "RMSE": np.sqrt(mean_squared_error(
            group["actual"], group["predicted"]
        )),
        "max_abs_error": group["abs_error"].max(),
        "mean_bias": group["error"].mean()
    })

pooled_results_df = pd.DataFrame(pooled_results)

print("\nPOOLED OUT-OF-FOLD RESULTS")
display(pooled_results_df.round(4))


# ============================================================
# 8. COMPARE AGAINST BASELINE
# ============================================================

baseline = pooled_results_df.loc[
    pooled_results_df["model"] == "M0"
].iloc[0]

comparison = pooled_results_df.copy()

comparison["MAE_improvement_pct"] = (
    (baseline["MAE"] - comparison["MAE"])
    / baseline["MAE"] * 100
)

comparison["RMSE_improvement_pct"] = (
    (baseline["RMSE"] - comparison["RMSE"])
    / baseline["RMSE"] * 100
)

comparison = comparison.sort_values("MAE")

print("\nIMPROVEMENT OVER BASELINE")
display(comparison.round(4))

print("\nABLATION EXPERIMENT COMPLETE.")


FOLD-LEVEL RESULTS


,model,train_laps,test_laps,train_rows,test_rows,MAE,RMSE,max_abs_error
0,M0,<= 15,16-24,264,149,2.9942,4.5674,15.1631
1,M0,<= 24,25-32,413,26,3.9746,5.8158,15.1401
2,M0,<= 32,33-40,439,20,13.3042,13.3760,15.0755
3,M0,<= 40,41-46,459,108,2.6152,2.8916,6.2959
4,M0,<= 46,47-52,567,83,1.2490,1.5768,7.7209
5,M1a,<= 15,16-24,264,149,2.9805,4.5573,15.1500
6,M1a,<= 24,25-32,413,26,3.9250,5.7620,15.0669
7,M1a,<= 32,33-40,439,20,14.1413,14.2094,15.9113
8,M1a,<= 40,41-46,459,108,2.7718,3.0714,6.7393
9,M1a,<= 46,47-52,567,83,1.2722,1.5963,7.7162



POOLED OUT-OF-FOLD RESULTS


,model,observations,MAE,RMSE,max_abs_error,mean_bias
0,M0,386,3.1131,4.7408,15.1631,0.2102
1,M1a,386,3.1967,4.8890,15.9113,0.3146
2,M1b,386,3.1609,5.0250,17.2214,0.2130
3,M1c,386,3.1064,4.4526,18.1071,-0.1227
4,M1d,386,2.9418,4.6029,17.5867,-0.3187



IMPROVEMENT OVER BASELINE


,model,observations,MAE,RMSE,max_abs_error,mean_bias,MAE_improvement_pct,RMSE_improvement_pct
4,M1d,386,2.9418,4.6029,17.5867,-0.3187,5.5028,2.9092
3,M1c,386,3.1064,4.4526,18.1071,-0.1227,0.2166,6.0797
0,M0,386,3.1131,4.7408,15.1631,0.2102,0.0000,0.0000
2,M1b,386,3.1609,5.0250,17.2214,0.2130,-1.5357,-5.9942
1,M1a,386,3.1967,4.8890,15.9113,0.3146,-2.6841,-3.1258



ABLATION EXPERIMENT COMPLETE.


In [ ]:

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ============================================================
# 1. PREPARE DATA
# ============================================================

df = df.copy()

df = df[
    (df["is_clean_lap"] == True)
    & (df["compound"].isin(["SOFT", "MEDIUM", "HARD"]))
].copy()

for col in ["lap_number", "lap_duration", "tyre_age"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(
    subset=[
        "lap_number", "lap_duration",
        "tyre_age", "driver_number", "compound"
    ]
).copy()

df["lap_number"] = df["lap_number"].astype(int)
df["compound"] = df["compound"].astype(str)

COMPOUNDS = ["SOFT", "MEDIUM", "HARD"]

# Compound-specific tyre-age interaction features
for compound in COMPOUNDS:
    df[f"age_{compound}"] = (
        df["tyre_age"] * (df["compound"] == compound).astype(int)
    )

print("Dataset shape:", df.shape)
print("Compound counts:")
display(df["compound"].value_counts())


# ============================================================
# 2. DEFINE THREE MODEL STRUCTURES
# ============================================================

base_categorical = ["driver_number", "compound"]

environment_numeric = [
    "track_temperature",
    "rainfall",
    "air_temperature",
    "wind_speed"
]

environment_categorical = ["grip_condition"]

model_configs = {

    # T0: One common tyre-age slope
    "T0_common_age": {
        "numeric": ["lap_number", "tyre_age"],
        "categorical": base_categorical,
        "slope_features": ["tyre_age"]
    },

    # T1: Separate slopes for each compound
    "T1_compound_age": {
        "numeric": [
            "lap_number",
            "age_SOFT",
            "age_MEDIUM",
            "age_HARD"
        ],
        "categorical": base_categorical,
        "slope_features": [
            "age_SOFT",
            "age_MEDIUM",
            "age_HARD"
        ]
    },

    # T2: Separate slopes + environmental features
    "T2_environment_age": {
        "numeric": [
            "lap_number",
            "age_SOFT",
            "age_MEDIUM",
            "age_HARD"
        ] + environment_numeric,
        "categorical": base_categorical + environment_categorical,
        "slope_features": [
            "age_SOFT",
            "age_MEDIUM",
            "age_HARD"
        ]
    }
}


# ============================================================
# 3. CHRONOLOGICAL VALIDATION FOLDS
# ============================================================

folds = [
    (15, 24),
    (24, 32),
    (32, 40),
    (40, 46),
    (46, 52)
]


# ============================================================
# 4. PIPELINE
# ============================================================

def build_model(numeric_features, categorical_features):

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessing = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    return Pipeline([
        ("preprocessing", preprocessing),
        ("regressor", Ridge(alpha=1.0))
    ])


# ============================================================
# 5. RUN VALIDATION
# ============================================================

fold_results = []
oof_predictions = []
fold_slopes = []

for model_name, config in model_configs.items():

    numeric_features = config["numeric"]
    categorical_features = config["categorical"]
    features = numeric_features + categorical_features

    for train_end, test_end in folds:

        train = df[df["lap_number"] <= train_end].copy()

        test = df[
            (df["lap_number"] > train_end)
            & (df["lap_number"] <= test_end)
        ].copy()

        if train.empty or test.empty:
            continue

        X_train = train[features].copy()
        X_test = test[features].copy()

        for col in categorical_features:
            X_train[col] = X_train[col].fillna("UNKNOWN").astype(str)
            X_test[col] = X_test[col].fillna("UNKNOWN").astype(str)

        y_train = train["lap_duration"].to_numpy()
        y_test = test["lap_duration"].to_numpy()

        model = build_model(
            numeric_features,
            categorical_features
        )

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        errors = predictions - y_test

        fold_results.append({
            "model": model_name,
            "train_laps": f"<= {train_end}",
            "test_laps": f"{train_end + 1}-{test_end}",
            "train_rows": len(train),
            "test_rows": len(test),
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
            "max_abs_error": np.max(np.abs(errors))
        })

        oof_predictions.append(pd.DataFrame({
            "model": model_name,
            "lap_number": test["lap_number"].to_numpy(),
            "actual": y_test,
            "predicted": predictions,
            "error": errors,
            "abs_error": np.abs(errors)
        }))

        # Recover slopes in original seconds-per-tyre-age-lap units
        preprocessing = model.named_steps["preprocessing"]
        scaler = preprocessing.named_transformers_[
            "numeric"
        ].named_steps["scaler"]

        coefficients = model.named_steps["regressor"].coef_
        numeric_scales = scaler.scale_

        for feature in config["slope_features"]:

            feature_index = numeric_features.index(feature)

            original_slope = (
                coefficients[feature_index]
                / numeric_scales[feature_index]
            )

            fold_slopes.append({
                "model": model_name,
                "train_laps": f"<= {train_end}",
                "test_laps": f"{train_end + 1}-{test_end}",
                "feature": feature,
                "slope_seconds_per_age_lap": original_slope
            })


# ============================================================
# 6. FOLD-LEVEL PERFORMANCE
# ============================================================

fold_results_df = pd.DataFrame(fold_results)

print("\nFOLD-LEVEL PERFORMANCE")
display(fold_results_df.round(4))


# ============================================================
# 7. POOLED OUT-OF-FOLD PERFORMANCE
# ============================================================

predictions_df = pd.concat(
    oof_predictions,
    ignore_index=True
)

pooled_results = []

for model_name, group in predictions_df.groupby("model"):

    pooled_results.append({
        "model": model_name,
        "observations": len(group),
        "MAE": mean_absolute_error(
            group["actual"], group["predicted"]
        ),
        "RMSE": np.sqrt(mean_squared_error(
            group["actual"], group["predicted"]
        )),
        "max_abs_error": group["abs_error"].max(),
        "mean_bias": group["error"].mean()
    })

pooled_results_df = pd.DataFrame(pooled_results)

print("\nPOOLED OUT-OF-FOLD PERFORMANCE")
display(pooled_results_df.round(4))


# ============================================================
# 8. TYRE-AGE SLOPE STABILITY ACROSS FOLDS
# ============================================================

fold_slopes_df = pd.DataFrame(fold_slopes)

print("\nTYRE-AGE SLOPES BY VALIDATION FOLD")
display(fold_slopes_df.round(5))


# ============================================================
# 9. FIT ON FULL DATA FOR DESCRIPTIVE SLOPE ESTIMATES
# ============================================================

full_data_slopes = []

for model_name, config in model_configs.items():

    numeric_features = config["numeric"]
    categorical_features = config["categorical"]
    features = numeric_features + categorical_features

    X = df[features].copy()

    for col in categorical_features:
        X[col] = X[col].fillna("UNKNOWN").astype(str)

    y = df["lap_duration"].to_numpy()

    model = build_model(
        numeric_features,
        categorical_features
    )

    model.fit(X, y)

    preprocessing = model.named_steps["preprocessing"]
    scaler = preprocessing.named_transformers_[
        "numeric"
    ].named_steps["scaler"]

    coefficients = model.named_steps["regressor"].coef_

    for feature in config["slope_features"]:

        idx = numeric_features.index(feature)

        slope = coefficients[idx] / scaler.scale_[idx]

        full_data_slopes.append({
            "model": model_name,
            "feature": feature,
            "slope_seconds_per_age_lap": slope
        })

full_slopes_df = pd.DataFrame(full_data_slopes)

print("\nFULL-DATA TYRE-AGE SLOPE ESTIMATES")
display(full_slopes_df.round(5))

print("\nEXPERIMENT COMPLETE.")


Dataset shape: (650, 36)
Compound counts:


compound
MEDIUM    461
SOFT      151
HARD       38
Name: count, dtype: int64


FOLD-LEVEL PERFORMANCE


,model,train_laps,test_laps,train_rows,test_rows,MAE,RMSE,max_abs_error
0,T0_common_age,<= 15,16-24,264,149,2.9942,4.5674,15.1631
1,T0_common_age,<= 24,25-32,413,26,3.9746,5.8158,15.1401
2,T0_common_age,<= 32,33-40,439,20,13.3042,13.3760,15.0755
3,T0_common_age,<= 40,41-46,459,108,2.6152,2.8916,6.2959
4,T0_common_age,<= 46,47-52,567,83,1.2490,1.5768,7.7209
5,T1_compound_age,<= 15,16-24,264,149,3.1145,4.6530,15.3371
6,T1_compound_age,<= 24,25-32,413,26,4.0104,5.8550,15.1882
7,T1_compound_age,<= 32,33-40,439,20,13.3785,13.4452,15.0173
8,T1_compound_age,<= 40,41-46,459,108,2.5699,2.8511,6.2889
9,T1_compound_age,<= 46,47-52,567,83,0.9518,1.3960,7.0692



POOLED OUT-OF-FOLD PERFORMANCE


,model,observations,MAE,RMSE,max_abs_error,mean_bias
0,T0_common_age,386,3.1131,4.7408,15.1631,0.2102
1,T1_compound_age,386,3.0892,4.7672,15.3371,0.0343
2,T2_environment_age,386,2.9924,4.6103,17.5928,-0.4218



TYRE-AGE SLOPES BY VALIDATION FOLD


,model,train_laps,test_laps,feature,slope_seconds_per_age_lap
0,T0_common_age,<= 15,16-24,tyre_age,0.03060
1,T0_common_age,<= 24,25-32,tyre_age,-0.05069
2,T0_common_age,<= 32,33-40,tyre_age,-0.14576
3,T0_common_age,<= 40,41-46,tyre_age,0.16247
4,T0_common_age,<= 46,47-52,tyre_age,0.22214
5,T1_compound_age,<= 15,16-24,age_SOFT,0.20027
6,T1_compound_age,<= 15,16-24,age_MEDIUM,0.00734
7,T1_compound_age,<= 15,16-24,age_HARD,0.04352
8,T1_compound_age,<= 24,25-32,age_SOFT,0.06499
9,T1_compound_age,<= 24,25-32,age_MEDIUM,-0.06693



FULL-DATA TYRE-AGE SLOPE ESTIMATES


,model,feature,slope_seconds_per_age_lap
0,T0_common_age,tyre_age,0.22132
1,T1_compound_age,age_SOFT,0.09838
2,T1_compound_age,age_MEDIUM,0.23736
3,T1_compound_age,age_HARD,0.09294
4,T2_environment_age,age_SOFT,0.08446
5,T2_environment_age,age_MEDIUM,0.11285
6,T2_environment_age,age_HARD,0.03899



EXPERIMENT COMPLETE.


In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 1. PREPARE DATA
# ============================================================

df = df.copy()

required_cols = [
    "driver_number", "lap_number", "lap_duration",
    "compound", "tyre_age", "stint_number"
]

missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Keep clean, usable laps
df = df[
    (df["is_clean_lap"] == True) &
    df["lap_duration"].notna() &
    df["tyre_age"].notna() &
    df["stint_number"].notna() &
    df["compound"].notna()
].copy()

df["compound"] = df["compound"].str.upper()

# Include dry compounds and Intermediate
compounds = ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE"]
df = df[df["compound"].isin(compounds)].copy()

# Unique stint identifier
df["stint_id"] = (
    df["driver_number"].astype(str) + "_" +
    df["stint_number"].astype(str)
)

print("CLEAN DATA SHAPE:", df.shape)
print("\nCOMPOUND COUNTS:")
print(df["compound"].value_counts())


# ============================================================
# 2. STINT COVERAGE
# ============================================================

stint_summary = (
    df.groupby(["compound", "stint_id"])
    .agg(
        clean_laps=("lap_number", "count"),
        min_tyre_age=("tyre_age", "min"),
        max_tyre_age=("tyre_age", "max"),
        age_span=("tyre_age", lambda x: x.max() - x.min()),
        first_lap=("lap_number", "min"),
        last_lap=("lap_number", "max")
    )
    .reset_index()
)

compound_coverage = (
    stint_summary.groupby("compound")
    .agg(
        total_stints=("stint_id", "count"),
        total_clean_laps=("clean_laps", "sum"),
        median_laps_per_stint=("clean_laps", "median"),
        median_age_span=("age_span", "median"),
        stints_4plus_laps=("clean_laps", lambda x: (x >= 4).sum()),
        stints_3plus_age_span=("age_span", lambda x: (x >= 3).sum())
    )
    .sort_values("total_clean_laps", ascending=False)
)

print("\n" + "=" * 65)
print("STINT COVERAGE BY COMPOUND")
print("=" * 65)
display(compound_coverage.round(2))


# ============================================================
# 3. WITHIN-STINT LAP TIME RELATIONSHIP
# ============================================================

# Only retain stints with enough observations and age variation
eligible_stints = stint_summary[
    (stint_summary["clean_laps"] >= 4) &
    (stint_summary["age_span"] >= 3)
]["stint_id"]

within_df = df[df["stint_id"].isin(eligible_stints)].copy()

# Demean lap duration and tyre age within each stint
within_df["lap_time_centered"] = (
    within_df["lap_duration"] -
    within_df.groupby("stint_id")["lap_duration"].transform("mean")
)

within_df["tyre_age_centered"] = (
    within_df["tyre_age"] -
    within_df.groupby("stint_id")["tyre_age"].transform("mean")
)

# Calculate descriptive within-stint slope for each compound
def within_stint_slope(group):
    x = group["tyre_age_centered"]
    y = group["lap_time_centered"]

    denominator = (x ** 2).sum()

    if denominator == 0:
        return np.nan

    return (x * y).sum() / denominator


within_stint_results = (
    within_df.groupby("compound")
    .apply(within_stint_slope, include_groups=False)
    .rename("within_stint_slope_sec_per_tyre_lap")
    .to_frame()
)

eligible_counts = (
    within_df.groupby("compound")
    .agg(
        eligible_stints=("stint_id", "nunique"),
        observations=("lap_duration", "count")
    )
)

within_stint_results = within_stint_results.join(eligible_counts)

print("\n" + "=" * 65)
print("WITHIN-STINT TYRE AGE RELATIONSHIP")
print("=" * 65)
display(within_stint_results.round(4))


# ============================================================
# 4. INTERMEDIATE PERFORMANCE UNDER DIFFERENT CONDITIONS
# ============================================================

intermediate = df[df["compound"] == "INTERMEDIATE"].copy()

if not intermediate.empty:

    print("\n" + "=" * 65)
    print("INTERMEDIATE: TRACK CONDITION DISTRIBUTION")
    print("=" * 65)

    for col in ["rainfall", "grip_condition"]:
        if col in intermediate.columns:
            print(f"\n{col.upper()}:")
            print(intermediate[col].value_counts(dropna=False))

    # Tyre age bands
    intermediate["age_band"] = pd.cut(
        intermediate["tyre_age"],
        bins=[-1, 5, 10, 15, 20, 30, 100],
        labels=["1-5", "6-10", "11-15", "16-20", "21-30", "31+"]
    )

    condition_cols = [
        col for col in ["rainfall", "grip_condition"]
        if col in intermediate.columns
    ]

    for condition in condition_cols:

        print(f"\nINTERMEDIATE BY TYRE AGE AND {condition.upper()}")

        summary = (
            intermediate.groupby(
                ["age_band", condition],
                observed=True,
                dropna=False
            )
            .agg(
                observations=("lap_duration", "count"),
                median_lap_time=("lap_duration", "median"),
                median_tyre_age=("tyre_age", "median")
            )
            .reset_index()
        )

        display(summary.round(3))

else:
    print("\nNo clean Intermediate observations available.")


# ============================================================
# 5. STINT-LEVEL DETAIL
# ============================================================

print("\n" + "=" * 65)
print("STINT-LEVEL DETAIL")
print("=" * 65)

display(
    stint_summary.sort_values(
        ["compound", "first_lap"]
    ).round(2)
)

CLEAN DATA SHAPE: (650, 37)

COMPOUND COUNTS:
compound
MEDIUM    461
SOFT      151
HARD       38
Name: count, dtype: int64

STINT COVERAGE BY COMPOUND


,total_stints,total_clean_laps,median_laps_per_stint,median_age_span,stints_4plus_laps,stints_3plus_age_span
compound,,,,,,
MEDIUM,24,461,24.0,23.0,23,23
SOFT,13,151,12.0,11.0,12,12
HARD,3,38,12.0,11.0,3,3



WITHIN-STINT TYRE AGE RELATIONSHIP


,within_stint_slope_sec_per_tyre_lap,eligible_stints,observations
compound,,,
HARD,0.0592,3,38
MEDIUM,0.1858,23,458
SOFT,0.0629,12,149



No clean Intermediate observations available.

STINT-LEVEL DETAIL


,compound,stint_id,clean_laps,min_tyre_age,max_tyre_age,age_span,first_lap,last_lap
0,HARD,11_1.0,17,1.0,17.0,16.0,2,18
1,HARD,1_3.0,12,1.0,12.0,11.0,40,51
2,HARD,55_3.0,9,1.0,9.0,8.0,41,49
4,MEDIUM,14_1.0,25,1.0,25.0,24.0,2,26
6,MEDIUM,16_1.0,17,1.0,17.0,16.0,2,18
7,MEDIUM,18_1.0,24,1.0,24.0,23.0,2,25
9,MEDIUM,1_1.0,24,1.0,24.0,23.0,2,25
10,MEDIUM,20_1.0,25,1.0,25.0,24.0,2,26
11,MEDIUM,22_1.0,25,1.0,25.0,24.0,2,26
12,MEDIUM,23_1.0,25,1.0,25.0,24.0,2,26


In [ ]:
import statsmodels.formula.api as smf

# Dry compounds only
model_df = df[df["compound"].isin(
    ["SOFT", "MEDIUM", "HARD"]
)].copy()

# Separate tyre-age effect by compound
formula = """
lap_duration ~
C(lap_number) +
C(driver_number) +
C(compound) +
tyre_age:C(compound)
"""

model = smf.ols(formula, data=model_df).fit(
    cov_type="cluster",
    cov_kwds={"groups": model_df["stint_id"]}
)

print(model.summary())

print("\nCOMPOUND-SPECIFIC TYRE AGE COEFFICIENTS")

for compound in ["SOFT", "MEDIUM", "HARD"]:
    term = f"tyre_age:C(compound)[{compound}]"

    if term in model.params.index:
        print(
            compound,
            "slope =", round(model.params[term], 4),
            "p-value =", round(model.pvalues[term], 4),
            "CI =", model.conf_int().loc[term].round(4).tolist()
        )

                            OLS Regression Results                            
Dep. Variable:           lap_duration   R-squared:                       0.936
Model:                            OLS   Adj. R-squared:                  0.930
Method:                 Least Squares   F-statistic:                -3.110e+13
Date:                Sat, 31 Oct 2026   Prob (F-statistic):               1.00
Time:                        13:56:33   Log-Likelihood:                -737.99
No. Observations:                 650   AIC:                             1600.
Df Residuals:                     588   BIC:                             1878.
Df Model:                          61                                         
Covariance Type:              cluster                                         
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

c:\Users\Nihan\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\statsmodels\base\model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 61, but rank is 39
  res = self.wald_test(


In [ ]:
import pitwall.engine as engine

# Baseline: keep the current tyres and make no further pit stops
stay_out_result = engine.replay(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    strategy=engine.Stay(),
    pace_model=production_pace_model,
    pit_loss=22.0,
    track_mode="causal",
    race_length=52
)

# Display results
print("STAY OUT BASELINE")
print("=" * 50)

display(stay_out_result.summary)
display(stay_out_result.laps_table.tail(10))

STAY OUT BASELINE


{'target_driver': 44,
 'decision_lap': 26,
 'race_length': 52,
 'ground_truth_available': False,
 'final_position': 12,
 'final_cumulative_time': 4915.29523874743,
 'total_time_delta_vs_actual': None,
 'positions_gained_vs_actual': None}

,lap_number,driver_number,lap_duration,cum_time,position,gap_to_leader,gap_ahead,gap_behind,compound,tyre_age,is_target,ground_truth_available
922,52,3,99.223575,4910.230213,10,758.780927,0.943980,0.272276,MEDIUM,50,False,False
923,52,4,97.110701,4910.502489,11,759.053203,0.272276,4.792749,MEDIUM,51,False,False
924,52,44,97.134268,4915.295239,12,763.845953,4.792749,5.758097,MEDIUM,51,True,False
925,52,63,97.271733,4921.053335,13,769.604049,5.758097,14.630094,MEDIUM,51,False,False
926,52,77,99.747737,4935.683429,14,784.234144,14.630094,34.320074,MEDIUM,50,False,False
927,52,14,98.207778,4970.003503,15,818.554218,34.320074,2.806892,MEDIUM,51,False,False
928,52,23,98.253697,4972.810395,16,821.361109,2.806892,1.927330,MEDIUM,51,False,False
929,52,22,98.351133,4974.737725,17,823.288439,1.927330,1.748362,MEDIUM,51,False,False
930,52,20,98.378647,4976.486087,18,825.036801,1.748362,2.780638,MEDIUM,51,False,False
931,52,2,98.364133,4979.266725,19,827.817439,2.780638,0.000000,MEDIUM,51,False,False


In [ ]:
target_projection = stay_out_result.laps_table[
    (stay_out_result.laps_table["driver_number"] == 44) &
    (stay_out_result.laps_table["lap_number"] > 26)
].copy()

display(
    target_projection[
        [
            "lap_number",
            "lap_duration",
            "cum_time",
            "position",
            "compound",
            "tyre_age",
            "ground_truth_available"
        ]
    ].head(15)
)

print("Projected laps:", len(target_projection))
print("Average projected lap time:", round(target_projection["lap_duration"].mean(), 3))

,lap_number,lap_duration,cum_time,position,compound,tyre_age,ground_truth_available
450,27,101.781366,2431.173366,13,MEDIUM,26,False
469,28,101.595482,2532.768848,13,MEDIUM,27,False
488,29,101.409598,2634.178445,13,MEDIUM,28,False
507,30,101.223714,2735.402159,13,MEDIUM,29,False
526,31,101.037830,2836.439990,13,MEDIUM,30,False
545,32,100.851946,2937.291936,13,MEDIUM,31,False
564,33,100.666062,3037.957998,13,MEDIUM,32,False
583,34,100.480178,3138.438176,13,MEDIUM,33,False
602,35,100.294294,3238.732471,13,MEDIUM,34,False
621,36,100.108411,3338.840881,13,MEDIUM,35,False


Projected laps: 26
Average projected lap time: 99.458


In [ ]:
print("PRODUCTION PACE MODEL")
print("=" * 50)

print("Model type:", type(production_pace_model))

print("\nAvailable attributes:")
print([
    attr for attr in dir(production_pace_model)
    if not attr.startswith("_")
])

print("\nCompound age slopes:")
print(production_pace_model.compound_age_slopes)

print("\nProjected lap-time trend:")
display(
    target_projection.groupby("compound")["lap_duration"]
    .agg(["count", "min", "max", "mean"])
)

PRODUCTION PACE MODEL
Model type: <class 'pitwall.pace_model.PaceModel'>

Available attributes:
['base_driver', 'base_lap', 'coefficients', 'compound_age_slopes', 'compound_effects', 'driver_effects', 'fit', 'fitted', 'identification_report', 'intercept', 'lap_effects', 'predict_lap', 'reference_compound', 'rsquared']

Compound age slopes:
{'MEDIUM': -0.18588390842579838, 'HARD': -0.18485146832367605, 'INTERMEDIATE': 0.0, 'SOFT': -0.0844933928877537}

Projected lap-time trend:


,count,min,max,mean
compound,,,,
MEDIUM,26,97.134268,101.781366,99.457817


In [ ]:
# Compare current model against a zero-degradation diagnostic

import copy

no_degradation_model = copy.deepcopy(production_pace_model)

no_degradation_model.compound_age_slopes = {
    compound: 0.0
    for compound in production_pace_model.compound_age_slopes
}

# Predict the same future laps using both models
comparison = []

for lap in range(27, 53):
    tyre_age = lap - 1

    current_prediction = production_pace_model.predict_lap(
        lap_number=lap,
        driver_number=44,
        compound="MEDIUM",
        tyre_age=tyre_age,
        mode="causal",
        decision_lap=26
    )

    baseline_prediction = no_degradation_model.predict_lap(
        lap_number=lap,
        driver_number=44,
        compound="MEDIUM",
        tyre_age=tyre_age,
        mode="causal",
        decision_lap=26
    )

    comparison.append({
        "lap_number": lap,
        "tyre_age": tyre_age,
        "current_model": current_prediction,
        "zero_degradation": baseline_prediction
    })

comparison_df = pd.DataFrame(comparison)

display(comparison_df.round(3))

print(
    "Average current prediction:",
    round(comparison_df["current_model"].mean(), 3)
)

print(
    "Average zero-degradation prediction:",
    round(comparison_df["zero_degradation"].mean(), 3)
)

,lap_number,tyre_age,current_model,zero_degradation
0,27,26,101.781,106.614
1,28,27,101.595,106.614
2,29,28,101.410,106.614
3,30,29,101.224,106.614
4,31,30,101.038,106.614
5,32,31,100.852,106.614
6,33,32,100.666,106.614
7,34,33,100.480,106.614
8,35,34,100.294,106.614
9,36,35,100.108,106.614


Average current prediction: 99.458
Average zero-degradation prediction: 106.614


In [7]:
import copy

# Start fresh from the original production model
calibrated_pace_model = copy.deepcopy(production_pace_model)

# Positive degradation estimates from our Ridge experiment
POSITIVE_DEGRADATION = {
    "SOFT": 0.09838,
    "MEDIUM": 0.23736,
    "HARD": 0.09294,
    "INTERMEDIATE": 0.0
}

REFERENCE_AGE = 25

# Recalibrate compound effects individually
for compound, new_slope in POSITIVE_DEGRADATION.items():

    old_slope = production_pace_model.compound_age_slopes.get(
        compound, 0.0
    )

    old_effect = production_pace_model.compound_effects.get(
        compound, 0.0
    )

    calibrated_pace_model.compound_effects[compound] = (
        old_effect + (old_slope - new_slope) * REFERENCE_AGE
    )

# Replace degradation slopes
calibrated_pace_model.compound_age_slopes = POSITIVE_DEGRADATION.copy()

# Verify reference-age predictions
print("REFERENCE-AGE CALIBRATION CHECK")
print("=" * 50)

for compound in ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE"]:

    old_prediction = production_pace_model.predict_lap(
        26, 44, compound, REFERENCE_AGE,
        mode="causal", decision_lap=26
    )

    new_prediction = calibrated_pace_model.predict_lap(
        26, 44, compound, REFERENCE_AGE,
        mode="causal", decision_lap=26
    )

    print(
        f"{compound:15s} | "
        f"Original: {old_prediction:.3f} | "
        f"Calibrated: {new_prediction:.3f} | "
        f"Difference: {new_prediction - old_prediction:.6f}"
    )

assert all(
    abs(
        calibrated_pace_model.predict_lap(
            26, 44, compound, REFERENCE_AGE,
            mode="causal", decision_lap=26
        )
        -
        production_pace_model.predict_lap(
            26, 44, compound, REFERENCE_AGE,
            mode="causal", decision_lap=26
        )
    ) < 1e-8
    for compound in POSITIVE_DEGRADATION
)

print("\nCalibration check passed.")

REFERENCE-AGE CALIBRATION CHECK
SOFT            | Original: 102.893 | Calibrated: 102.893 | Difference: 0.000000
MEDIUM          | Original: 100.435 | Calibrated: 100.435 | Difference: -0.000000
HARD            | Original: 101.144 | Calibrated: 101.144 | Difference: 0.000000
INTERMEDIATE    | Original: 100.106 | Calibrated: 100.106 | Difference: 0.000000

Calibration check passed.


In [ ]:
calibrated_stay_out_result = engine.replay(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    strategy=engine.Stay(),
    pace_model=calibrated_pace_model,
    pit_loss=22.0,
    track_mode="causal",
    race_length=52
)

print("CALIBRATED STAY-OUT BASELINE")
print("=" * 50)

display(calibrated_stay_out_result.summary)

target_laps = calibrated_stay_out_result.laps_table[
    (calibrated_stay_out_result.laps_table["driver_number"] == 44) &
    (calibrated_stay_out_result.laps_table["lap_number"] > 26)
]

display(
    target_laps[
        ["lap_number", "lap_duration", "position",
         "compound", "tyre_age"]
    ].head(15)
)

CALIBRATED STAY-OUT BASELINE


{'target_driver': 44,
 'decision_lap': 26,
 'race_length': 52,
 'ground_truth_available': False,
 'final_position': 12,
 'final_cumulative_time': 5063.853850604884,
 'total_time_delta_vs_actual': None,
 'positions_gained_vs_actual': None}

,lap_number,lap_duration,position,compound,tyre_age
450,27,102.20461,13,MEDIUM,26
469,28,102.44197,13,MEDIUM,27
488,29,102.67933,13,MEDIUM,28
507,30,102.91669,13,MEDIUM,29
526,31,103.15405,13,MEDIUM,30
545,32,103.39141,13,MEDIUM,31
564,33,103.62877,13,MEDIUM,32
583,34,103.86613,13,MEDIUM,33
602,35,104.10349,13,MEDIUM,34
621,36,104.34085,13,MEDIUM,35


In [8]:
# Inspect calibrated MEDIUM predictions and components

pm = calibrated_pace_model

for lap in [26, 27, 35, 40, 45, 52]:

    age = lap - 1

    prediction = pm.predict_lap(
        lap_number=lap,
        driver_number=44,
        compound="MEDIUM",
        tyre_age=age,
        mode="causal",
        decision_lap=26
    )

    print(
        f"Lap {lap:2d} | "
        f"Tyre age {age:2d} | "
        f"Predicted pace: {prediction:.3f}s"
    )

print("\nCalibrated model coefficients:")
print("Intercept:", pm.intercept)
print("MEDIUM compound effect:", pm.compound_effects["MEDIUM"])
print("MEDIUM age slope:", pm.compound_age_slopes["MEDIUM"])

Lap 26 | Tyre age 25 | Predicted pace: 100.435s
Lap 27 | Tyre age 26 | Predicted pace: 100.673s
Lap 35 | Tyre age 34 | Predicted pace: 102.572s
Lap 40 | Tyre age 39 | Predicted pace: 103.758s
Lap 45 | Tyre age 44 | Predicted pace: 104.945s
Lap 52 | Tyre age 51 | Predicted pace: 106.607s

Calibrated model coefficients:
Intercept: 92.42477196191189
MEDIUM compound effect: -9.935246020797402
MEDIUM age slope: 0.23736


In [12]:
strategy_results = optimize_strategy(
    session_df=df,
    target_driver=44,
    decision_lap=26,
    race_length=52,
    pace_model=calibrated_pace_model,
    pit_loss=22.0,
    track_mode="causal",
    max_stops=2,
    max_candidates=95,
    supported_compounds={"SOFT", "MEDIUM", "HARD"}
)

print(strategy_results.ranking.head(15))

TypeError: optimize_strategy() got an unexpected keyword argument 'race_length'

## 3. Empirical Pit-Loss Measurement

Pit loss is measured directly by comparing actual in-lap + out-lap times against expected clean pace from the model.

In [ ]:
green_summary, sc_summary = measure_pit_loss(df, model)
print(f'Green-flag pit stops measured: n={green_summary.n_stops}')
print(f'Median pit loss: {green_summary.median:.2f} s (IQR: [{green_summary.iqr_low:.2f}, {green_summary.iqr_high:.2f}] s)')
print(f'Mean pit loss: {green_summary.mean:.2f} s +/- {green_summary.std:.2f} s')

# Plot pit loss distribution
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(green_summary.stops_table['pit_loss'], bins=15, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(green_summary.median, color='red', linestyle='--', linewidth=2, label=f'Median ({green_summary.median:.2f}s)')
ax.axvline(22.0, color='orange', linestyle=':', linewidth=2, label='Old Assumption (22.0s)')
ax.set_title('Green-Flag Pit Loss Distribution (Session 9558)')
ax.set_xlabel('Pit Loss (s)')
ax.set_ylabel('Count')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Multi-Stop Strategy Candidate Generation

For driver 44 (Lewis Hamilton) at decision lap 26, procedural multi-stop candidates are generated and ranked by distance to historical baseline.

In [ ]:
cands_44 = generate_multi_stop_candidates(df, target_driver=44, decision_lap=26)
print(f'Total candidate count: {len(cands_44)}')
print('\nTop 10 candidates by baseline distance:')
display(cands_44[['strategy_id', 'pit_sequence_str', 'is_baseline', 'baseline_distance', 'num_stops']].head(10))

# Replaying the baseline candidate S001
base_row = cands_44[cands_44['is_baseline']].iloc[0]
strat_base = Strategy(pit_sequence=tuple(base_row['pit_sequence']), description=base_row['description'])
res_base = replay(df, 44, 26, strategy=strat_base, pace_model=model, pit_loss=green_summary.median)
print('\nBaseline candidate (S001) replay result:')
print(f'  Projected final position: P{res_base.summary["final_position"]} (Actual: P1)')
print(f'  Total time delta vs actual: {res_base.summary["total_time_delta_vs_actual"]:.2f} s')

## 5. Acceptance Test Suite (Tests A, B, C, D)

Test B evaluates finishers against their historical multi-stop baseline.

In [ ]:
res_a = run_plumbing_test(df)
res_b = run_identity_test(df, pace_model=model, pit_loss=green_summary.median)
res_c = run_sanity_bound_test(df, target_driver=44, decision_lap=20, pace_model=model, pit_loss=green_summary.median)
supp_ok, supp_msg = run_support_test(df, 'INTERMEDIATE', (28, 38))

test_results_summary = [
    {'Test': res_a.name, 'Status': 'PASS' if res_a.passed else 'FAIL', 'Metric': res_a.message},
    {'Test': res_b.name, 'Status': 'PASS' if res_b.passed else 'FAIL', 'Metric': res_b.message},
    {'Test': res_c.name, 'Status': 'PASS' if res_c.passed else 'FAIL', 'Metric': res_c.message},
    {'Test': 'Support Test (Test D)', 'Status': 'PASS' if supp_ok else 'FAIL', 'Metric': supp_msg}
]
display(pd.DataFrame(test_results_summary))

# Test B detailed table
tbl_b = pd.DataFrame(res_b.details['table'])
print('\nTest B Results Table (sample):')
display(tbl_b[['driver', 'decision_lap', 'pit_sequence', 'time_error_s', 'sim_final_pos', 'actual_final_pos', 'abs_pos_error']].head(15))

## 6. Gap-to-Leader vs Lap Comparison

Comparing Hamilton's actual race vs counterfactual pit strategies.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
res_stay = replay(df, 44, 26, strategy=Stay(), pace_model=model, pit_loss=green_summary.median)

ham_act = df[df['driver_number']==44]
ham_base = res_base.laps_table[res_base.laps_table['driver_number']==44]
ham_stay = res_stay.laps_table[res_stay.laps_table['driver_number']==44]

leader_times = df.groupby('lap_number')['cum_time'].min().to_dict()
act_gaps = [ham_act[ham_act['lap_number']==l]['cum_time'].iloc[0] - leader_times[l] for l in range(1, 53)]

ax.plot(range(1, 53), act_gaps, label='Actual Race (Hamilton)', color='purple', linewidth=2)
ax.plot(ham_base['lap_number'], ham_base['gap_to_leader'], label='Baseline Multi-Stop S001 (INTER 27 -> SOFT 38)', color='teal', linestyle='--', linewidth=2)
ax.plot(ham_stay['lap_number'], ham_stay['gap_to_leader'], label='Counterfactual Stay Out', color='red', linestyle=':', linewidth=2)
ax.set_title('Gap to Leader vs Lap: Hamilton Actual vs Multi-Stop Replay (Decision Lap 26)')
ax.set_xlabel('Lap Number')
ax.set_ylabel('Gap to Leader (s)')
ax.axvline(26, color='gray', linestyle='--', alpha=0.7, label='Decision Lap 26')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from pitwall.engine import Strategy, replay

# Scenario 1: Real history (what actually happened)
real = Strategy(
    pit_sequence=((27, 'INTERMEDIATE'), (38, 'SOFT')),
    description="Real: INTER lap 27 → SOFT lap 38"
)
result_real = replay(session_df, target_driver=44, decision_lap=25, strategy=real)

# Scenario 2: What if he stayed out?
stay_out = Strategy(
    pit_sequence=((999, 'MEDIUM'),),  # never pit
    description="Counterfactual: Stay out forever"
)
result_stay = replay(session_df, target_driver=44, decision_lap=25, strategy=stay_out)

# Compare
print("Real strategy (INTER 27 → SOFT 38):")
print(f"  Final position: {result_real.summary['final_position']}")
print(f"  Final time: {result_real.summary['final_time_s']:.2f} s")

print("\nStay out forever:")
print(f"  Final position: {result_stay.summary['final_position']}")
print(f"  Final time: {result_stay.summary['final_time_s']:.2f} s")

print(f"\nDifference: {result_stay.summary['final_time_s'] - result_real.summary['final_time_s']:.2f} s")